# Notebook 4: Baseline Modelling and Validation

## Purpose

This notebook develops and evaluates the first predictive models for next-year commodity-level food-supply shortage events.

The model receives information available in predictor year t and attempts to predict whether calorie availability for the same country and commodity will experience a qualifying shortage event in year t + 1.

The notebook will:

1. reload and validate the model-ready dataset created in Notebook 3;
2. explain and establish non-model and simple-model baselines;
3. preprocess missing, extreme, categorical and numerical values using training data only;
4. train interpretable baseline models;
5. compare performance using measures suitable for an imbalanced shortage target;
6. use the validation period for model and threshold decisions;
7. reserve the test period for one final unbiased evaluation.

The test data will not be used to select features, models, settings or probability thresholds.

In [1]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn

from IPython.display import display

from sklearn.base import (
    BaseEstimator,
    TransformerMixin,
    clone,
)
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

# Reproducibility
RANDOM_STATE = 42

# Display settings
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 240)
pd.set_option(
    "display.float_format",
    lambda value: f"{value:,.4f}",
)

# Project directories
PROJECT_DIR = (
    Path.home()
    / "Documents"
    / "food_security_predictor"
)

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "africa_first"
)

MODEL_OUTPUT_DIR = (
    PROJECT_DIR
    / "models"
    / "africa_first"
)

MODEL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Validated Notebook 3 inputs
MODEL_DATASET_PATH = (
    PROCESSED_DIR
    / "africa_model_ready_shortage_features_2010_2022.parquet"
)

FEATURE_REGISTRY_PATH = (
    PROCESSED_DIR
    / "shortage_feature_registry.csv"
)

FEATURE_SPECIFICATION_PATH = (
    PROCESSED_DIR
    / "shortage_feature_specification.json"
)

MODELLING_SPLIT_SUMMARY_PATH = (
    PROCESSED_DIR
    / "shortage_modelling_split_summary.csv"
)

PREPROCESSING_POLICY_PATH = (
    PROCESSED_DIR
    / "shortage_preprocessing_policy.csv"
)

required_files = [
    MODEL_DATASET_PATH,
    FEATURE_REGISTRY_PATH,
    FEATURE_SPECIFICATION_PATH,
    MODELLING_SPLIT_SUMMARY_PATH,
    PREPROCESSING_POLICY_PATH,
]

missing_files = [
    path.name
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following validated Notebook 3 files "
        "could not be found:\n"
        + "\n".join(
            f"- {name}"
            for name in missing_files
        )
    )

print("Notebook 4 environment prepared successfully.")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"Project directory: {PROJECT_DIR}")
print(f"Model output directory: {MODEL_OUTPUT_DIR}")

print("\nAll required Notebook 3 files were found:")

for path in required_files:
    print(f"- {path.name}")

Notebook 4 environment prepared successfully.
Scikit-learn version: 1.9.0
Project directory: /Users/adewale/Documents/food_security_predictor
Model output directory: /Users/adewale/Documents/food_security_predictor/models/africa_first

All required Notebook 3 files were found:
- africa_model_ready_shortage_features_2010_2022.parquet
- shortage_feature_registry.csv
- shortage_feature_specification.json
- shortage_modelling_split_summary.csv
- shortage_preprocessing_policy.csv


In [2]:
# ---------------------------------------------------------
# Load the validated Notebook 3 outputs
# ---------------------------------------------------------

model_data = pd.read_parquet(
    MODEL_DATASET_PATH
)

feature_registry = pd.read_csv(
    FEATURE_REGISTRY_PATH
)

with open(
    FEATURE_SPECIFICATION_PATH,
    "r",
    encoding="utf-8",
) as file:
    feature_specification = json.load(file)

saved_split_summary = pd.read_csv(
    MODELLING_SPLIT_SUMMARY_PATH
)

preprocessing_policy = pd.read_csv(
    PREPROCESSING_POLICY_PATH
)

# ---------------------------------------------------------
# Reconstruct the modelling definitions
# ---------------------------------------------------------

target_column = (
    feature_specification[
        "target_column"
    ]
)

feature_columns = (
    feature_specification[
        "final_feature_columns"
    ]
)

categorical_features = (
    feature_specification[
        "categorical_features"
    ]
)

binary_features = (
    feature_specification[
        "binary_indicator_features"
    ]
)

numeric_features = (
    feature_specification[
        "numeric_features"
    ]
)

panel_key_columns = (
    feature_specification[
        "panel_key_columns"
    ]
)

# ---------------------------------------------------------
# Confirm every specified feature exists
# ---------------------------------------------------------

missing_feature_columns = [
    column
    for column in feature_columns
    if column not in model_data.columns
]

unexpected_feature_columns = [
    column
    for column in (
        categorical_features
        + binary_features
        + numeric_features
    )
    if column not in feature_columns
]

if missing_feature_columns:
    raise KeyError(
        "Specified features missing from the "
        "model dataset:\n"
        + "\n".join(
            f"- {column}"
            for column in missing_feature_columns
        )
    )

if unexpected_feature_columns:
    raise ValueError(
        "Feature-group columns were found outside "
        "the final feature list:\n"
        + "\n".join(
            f"- {column}"
            for column in unexpected_feature_columns
        )
    )

# ---------------------------------------------------------
# Separate development data from the locked test period
# ---------------------------------------------------------

train_data = (
    model_data.loc[
        model_data["Temporal split"]
        .eq("Train")
    ]
    .copy()
)

validation_data = (
    model_data.loc[
        model_data["Temporal split"]
        .eq("Validation")
    ]
    .copy()
)

locked_test_data = (
    model_data.loc[
        model_data["Temporal split"]
        .eq("Test")
    ]
    .copy()
)

X_train = train_data[
    feature_columns
].copy()

y_train = train_data[
    target_column
].astype("int8").copy()

X_validation = validation_data[
    feature_columns
].copy()

y_validation = validation_data[
    target_column
].astype("int8").copy()

# The test predictors are prepared, but the test target
# is deliberately not assigned to a modelling variable yet.
X_test_locked = locked_test_data[
    feature_columns
].copy()

test_metadata_locked = locked_test_data[
    [
        "observation_id",
        "Area",
        "Item Code",
        "Item",
        "Year",
        "target_year",
        "Temporal split",
    ]
].copy()

# ---------------------------------------------------------
# Display the loaded checkpoint
# ---------------------------------------------------------

loaded_checkpoint_summary = pd.Series(
    {
        "Dataset rows": len(model_data),
        "Dataset columns": (
            model_data.shape[1]
        ),
        "Final predictors": (
            len(feature_columns)
        ),
        "Categorical predictors": (
            len(categorical_features)
        ),
        "Binary predictors": (
            len(binary_features)
        ),
        "Numeric predictors": (
            len(numeric_features)
        ),
        "Training rows": len(X_train),
        "Validation rows": (
            len(X_validation)
        ),
        "Locked test rows": (
            len(X_test_locked)
        ),
        "Training positive events": (
            y_train.sum()
        ),
        "Validation positive events": (
            y_validation.sum()
        ),
        "Duplicate observation IDs": (
            model_data[
                "observation_id"
            ].duplicated().sum()
        ),
        "Future information used": (
            feature_specification[
                "future_information_used"
            ]
        ),
    },
    name="Result",
).to_frame()

print("Loaded Notebook 3 modelling checkpoint:")
display(loaded_checkpoint_summary)

print("\nSaved chronological split definition:")
display(saved_split_summary)

print("\nSaved preprocessing policy:")
display(preprocessing_policy)

# ---------------------------------------------------------
# Audit feature availability in development data
# ---------------------------------------------------------

development_availability = pd.DataFrame(
    {
        "Feature group": [
            "Categorical",
            "Binary indicator",
            "Numeric",
        ],
        "Features": [
            len(categorical_features),
            len(binary_features),
            len(numeric_features),
        ],
        "Training missing cells": [
            X_train[
                categorical_features
            ].isna().sum().sum(),
            X_train[
                binary_features
            ].isna().sum().sum(),
            X_train[
                numeric_features
            ].isna().sum().sum(),
        ],
        "Validation missing cells": [
            X_validation[
                categorical_features
            ].isna().sum().sum(),
            X_validation[
                binary_features
            ].isna().sum().sum(),
            X_validation[
                numeric_features
            ].isna().sum().sum(),
        ],
    }
)

development_availability[
    "Training possible cells"
] = (
    development_availability[
        "Features"
    ]
    * len(X_train)
)

development_availability[
    "Validation possible cells"
] = (
    development_availability[
        "Features"
    ]
    * len(X_validation)
)

development_availability[
    "Training missing %"
] = (
    100
    * development_availability[
        "Training missing cells"
    ]
    / development_availability[
        "Training possible cells"
    ]
)

development_availability[
    "Validation missing %"
] = (
    100
    * development_availability[
        "Validation missing cells"
    ]
    / development_availability[
        "Validation possible cells"
    ]
)

print("\nDevelopment-period feature availability:")
display(development_availability)

# ---------------------------------------------------------
# Confirm the feature registry matches the specification
# ---------------------------------------------------------

registry_retained_features = (
    feature_registry.loc[
        feature_registry["Status"]
        .eq("Retained"),
        "Feature",
    ]
    .tolist()
)

# The manually excluded residual percentage feature
# has a non-retained status and must not appear here.
registry_final_set = set(
    registry_retained_features
)

specified_final_set = set(
    feature_columns
)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert model_data.shape == (
    3248,
    138,
)

assert len(feature_columns) == 129
assert len(categorical_features) == 3
assert len(binary_features) == 20
assert len(numeric_features) == 106

assert len(X_train) == 2250
assert len(X_validation) == 500
assert len(X_test_locked) == 498

assert y_train.sum() == 231
assert y_validation.sum() == 52

assert (
    len(categorical_features)
    + len(binary_features)
    + len(numeric_features)
    == len(feature_columns)
)

assert len(
    set(feature_columns)
) == len(feature_columns)

assert target_column not in feature_columns
assert "target_year" not in feature_columns
assert "Temporal split" not in feature_columns

assert feature_specification[
    "future_information_used"
] is False

assert model_data[
    "observation_id"
].is_unique

assert (
    model_data[
        panel_key_columns
    ].duplicated().sum()
    == 0
)

assert list(X_train.columns) == feature_columns
assert list(X_validation.columns) == feature_columns
assert list(X_test_locked.columns) == feature_columns

assert set(X_train.index).isdisjoint(
    X_validation.index
)

assert set(X_train.index).isdisjoint(
    X_test_locked.index
)

assert set(X_validation.index).isdisjoint(
    X_test_locked.index
)

assert registry_final_set == specified_final_set

print(
    "\nThe Notebook 3 modelling checkpoint was "
    "reloaded and validated successfully."
)

print(
    "The test period remains locked and has not "
    "been used for model evaluation."
)

Loaded Notebook 3 modelling checkpoint:


,Result
Dataset rows,3248
Dataset columns,138
Final predictors,129
Categorical predictors,3
Binary predictors,20
Numeric predictors,106
Training rows,2250
Validation rows,500
Locked test rows,498
Training positive events,231



Saved chronological split definition:


,Temporal split,Rows,Predictors,Predictor year start,Predictor year end,Target year start,Target year end,Positive events,Negative outcomes,Event rate %,Missing predictor cells,Missing predictor %
0,Train,2250,129,2010,2018,2011,2019,231,2019,10.2667,39985,13.7761
1,Validation,500,129,2019,2020,2020,2021,52,448,10.4000,5356,8.3039
2,Test,498,129,2021,2022,2022,2023,51,447,10.2410,5083,7.9123



Saved preprocessing policy:


,Feature type,Planned treatment,Why
0,Categorical features,One-hot encoding learned from training data,Models require numerical representations of na...
1,Binary indicators,Retained as 0/1 values,Their existing values already have a direct in...
2,Ordinary numeric features,Scaling learned from training data where required,Prevents large-unit variables from dominating ...
3,Numeric features with missing values,"Imputation learned from training data, with mi...",Validation and test information must not deter...
4,Extreme numeric values,Training-derived lower and upper boundaries; r...,Prevents a few unusually large ratios from dom...



Development-period feature availability:


,Feature group,Features,Training missing cells,Validation missing cells,Training possible cells,Validation possible cells,Training missing %,Validation missing %
0,Categorical,3,0,0,6750,1500,0.0000,0.0000
1,Binary indicator,20,0,0,45000,10000,0.0000,0.0000
2,Numeric,106,39985,5356,238500,53000,16.7652,10.1057



The Notebook 3 modelling checkpoint was reloaded and validated successfully.
The test period remains locked and has not been used for model evaluation.


In [3]:
# ---------------------------------------------------------
# Explain the evaluation measures
# ---------------------------------------------------------

metric_guide = pd.DataFrame(
    {
        "Measure": [
            "Accuracy",
            "Balanced accuracy",
            "Precision",
            "Recall",
            "F1 score",
            "F2 score",
            "ROC-AUC",
            "PR-AUC",
            "Brier score",
        ],
        "Plain-English meaning": [
            "Percentage of all observations classified correctly",
            "Average of performance on shortages and non-shortages",
            "Of all shortage warnings issued, how many were correct",
            "Of all real shortages, how many the model detected",
            "Balance between precision and recall",
            "Balance that gives recall twice the importance of precision",
            "Ability to rank shortages above non-shortages across thresholds",
            "Quality of shortage ranking when shortage events are uncommon",
            "Average error in predicted probabilities; lower is better",
        ],
        "Preferred direction": [
            "Higher",
            "Higher",
            "Higher",
            "Higher",
            "Higher",
            "Higher",
            "Higher",
            "Higher",
            "Lower",
        ],
    }
)

print("Model-evaluation guide:")
display(metric_guide)

# ---------------------------------------------------------
# Reusable evaluation function
# ---------------------------------------------------------

def evaluate_probabilities(
    y_true,
    predicted_probability,
    threshold=0.50,
    model_name="Model",
    dataset_name="Validation",
):
    """
    Convert predicted probabilities into classifications
    and calculate shortage-relevant performance measures.
    """

    y_true_array = np.asarray(
        y_true,
        dtype=int,
    )

    probability_array = np.asarray(
        predicted_probability,
        dtype=float,
    )

    predicted_class = (
        probability_array >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true_array,
        predicted_class,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    result = {
        "Model": model_name,
        "Dataset": dataset_name,
        "Threshold": threshold,
        "Observations": len(y_true_array),
        "Actual events": int(
            y_true_array.sum()
        ),
        "Predicted events": int(
            predicted_class.sum()
        ),
        "Predicted event rate %": (
            100 * predicted_class.mean()
        ),
        "True negatives": int(tn),
        "False positives": int(fp),
        "False negatives": int(fn),
        "True positives": int(tp),
        "Accuracy": accuracy_score(
            y_true_array,
            predicted_class,
        ),
        "Balanced accuracy": (
            balanced_accuracy_score(
                y_true_array,
                predicted_class,
            )
        ),
        "Precision": precision_score(
            y_true_array,
            predicted_class,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_true_array,
            predicted_class,
            zero_division=0,
        ),
        "Specificity": specificity,
        "F1": f1_score(
            y_true_array,
            predicted_class,
            zero_division=0,
        ),
        "F2": fbeta_score(
            y_true_array,
            predicted_class,
            beta=2,
            zero_division=0,
        ),
        "ROC-AUC": roc_auc_score(
            y_true_array,
            probability_array,
        ),
        "PR-AUC": average_precision_score(
            y_true_array,
            probability_array,
        ),
        "Brier score": brier_score_loss(
            y_true_array,
            probability_array,
        ),
    }

    return result, predicted_class


# ---------------------------------------------------------
# Create the no-skill baseline
# ---------------------------------------------------------

training_event_rate = y_train.mean()

baseline_validation_probability = np.full(
    len(y_validation),
    training_event_rate,
    dtype=float,
)

baseline_validation_result, (
    baseline_validation_prediction
) = evaluate_probabilities(
    y_true=y_validation,
    predicted_probability=(
        baseline_validation_probability
    ),
    threshold=0.50,
    model_name=(
        "Always predict no shortage"
    ),
    dataset_name="Validation",
)

validation_evaluation_rows = [
    baseline_validation_result
]

baseline_validation_table = pd.DataFrame(
    [baseline_validation_result]
)

print("\nNon-model baseline performance:")
display(baseline_validation_table)

# ---------------------------------------------------------
# Display the confusion matrix in ordinary language
# ---------------------------------------------------------

baseline_confusion_table = pd.DataFrame(
    [
        {
            "Actual outcome": "No shortage",
            "Predicted no shortage": (
                baseline_validation_result[
                    "True negatives"
                ]
            ),
            "Predicted shortage": (
                baseline_validation_result[
                    "False positives"
                ]
            ),
        },
        {
            "Actual outcome": "Shortage",
            "Predicted no shortage": (
                baseline_validation_result[
                    "False negatives"
                ]
            ),
            "Predicted shortage": (
                baseline_validation_result[
                    "True positives"
                ]
            ),
        },
    ]
)

print("\nBaseline confusion matrix:")
display(baseline_confusion_table)

baseline_interpretation = pd.Series(
    {
        "Validation observations": (
            len(y_validation)
        ),
        "Real shortage events": (
            y_validation.sum()
        ),
        "Shortages detected": (
            baseline_validation_result[
                "True positives"
            ]
        ),
        "Shortages missed": (
            baseline_validation_result[
                "False negatives"
            ]
        ),
        "False warnings": (
            baseline_validation_result[
                "False positives"
            ]
        ),
        "Accuracy %": (
            100
            * baseline_validation_result[
                "Accuracy"
            ]
        ),
        "Recall %": (
            100
            * baseline_validation_result[
                "Recall"
            ]
        ),
        "Training event probability used %": (
            100 * training_event_rate
        ),
    },
    name="Result",
).to_frame()

print("\nBaseline interpretation:")
display(baseline_interpretation)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert baseline_validation_prediction.sum() == 0

assert (
    baseline_validation_result[
        "True negatives"
    ]
    == 448
)

assert (
    baseline_validation_result[
        "False positives"
    ]
    == 0
)

assert (
    baseline_validation_result[
        "False negatives"
    ]
    == 52
)

assert (
    baseline_validation_result[
        "True positives"
    ]
    == 0
)

assert np.isclose(
    baseline_validation_result[
        "Accuracy"
    ],
    448 / 500,
)

assert (
    baseline_validation_result[
        "Recall"
    ]
    == 0
)

assert np.isclose(
    baseline_validation_result[
        "ROC-AUC"
    ],
    0.5,
)

assert np.isclose(
    baseline_validation_result[
        "PR-AUC"
    ],
    y_validation.mean(),
)

print(
    "\nThe non-model baseline was established "
    "successfully."
)

print(
    "Its high accuracy is misleading because it "
    "failed to detect every validation shortage."
)

Model-evaluation guide:


,Measure,Plain-English meaning,Preferred direction
0,Accuracy,Percentage of all observations classified corr...,Higher
1,Balanced accuracy,Average of performance on shortages and non-sh...,Higher
2,Precision,"Of all shortage warnings issued, how many were...",Higher
3,Recall,"Of all real shortages, how many the model dete...",Higher
4,F1 score,Balance between precision and recall,Higher
5,F2 score,Balance that gives recall twice the importance...,Higher
6,ROC-AUC,Ability to rank shortages above non-shortages ...,Higher
7,PR-AUC,Quality of shortage ranking when shortage even...,Higher
8,Brier score,Average error in predicted probabilities; lowe...,Lower



Non-model baseline performance:


,Model,Dataset,Threshold,Observations,Actual events,Predicted events,Predicted event rate %,True negatives,False positives,False negatives,True positives,Accuracy,Balanced accuracy,Precision,Recall,Specificity,F1,F2,ROC-AUC,PR-AUC,Brier score
0,Always predict no shortage,Validation,0.5000,500,52,0,0.0000,448,0,52,0,0.8960,0.5000,0.0000,0.0000,1.0000,0.0000,0.0000,0.5000,0.1040,0.0932



Baseline confusion matrix:


,Actual outcome,Predicted no shortage,Predicted shortage
0,No shortage,448,0
1,Shortage,52,0



Baseline interpretation:


,Result
Validation observations,500.0000
Real shortage events,52.0000
Shortages detected,0.0000
Shortages missed,52.0000
False warnings,0.0000
Accuracy %,89.6000
Recall %,0.0000
Training event probability used %,10.2667



The non-model baseline was established successfully.
Its high accuracy is misleading because it failed to detect every validation shortage.


In [4]:
# ---------------------------------------------------------
# Custom training-fitted extreme-value limiter
# ---------------------------------------------------------

class QuantileClipper(
    BaseEstimator,
    TransformerMixin,
):
    """
    Limit numeric values to lower and upper quantiles
    learned only from the data supplied during fit().
    """

    def __init__(
        self,
        lower_quantile=0.01,
        upper_quantile=0.99,
    ):
        self.lower_quantile = lower_quantile
        self.upper_quantile = upper_quantile

    def fit(self, X, y=None):
        X_array = np.asarray(
            X,
            dtype=float,
        )

        if not (
            0 <= self.lower_quantile
            < self.upper_quantile
            <= 1
        ):
            raise ValueError(
                "Quantiles must satisfy "
                "0 <= lower < upper <= 1."
            )

        self.n_features_in_ = (
            X_array.shape[1]
        )

        self.fit_row_count_ = (
            X_array.shape[0]
        )

        with warnings.catch_warnings():
            warnings.simplefilter(
                "ignore",
                category=RuntimeWarning,
            )

            self.lower_bounds_ = (
                np.nanquantile(
                    X_array,
                    self.lower_quantile,
                    axis=0,
                )
            )

            self.upper_bounds_ = (
                np.nanquantile(
                    X_array,
                    self.upper_quantile,
                    axis=0,
                )
            )

        if np.isnan(
            self.lower_bounds_
        ).any() or np.isnan(
            self.upper_bounds_
        ).any():
            raise ValueError(
                "At least one numeric feature was "
                "entirely missing during preprocessing fit."
            )

        return self

    def transform(self, X):
        X_array = np.asarray(
            X,
            dtype=float,
        )

        if (
            X_array.shape[1]
            != self.n_features_in_
        ):
            raise ValueError(
                "The number of numeric features "
                "does not match the fitted data."
            )

        return np.clip(
            X_array,
            self.lower_bounds_,
            self.upper_bounds_,
        )

    def get_feature_names_out(
        self,
        input_features=None,
    ):
        if input_features is None:
            return np.asarray(
                [
                    f"x{index}"
                    for index in range(
                        self.n_features_in_
                    )
                ],
                dtype=object,
            )

        return np.asarray(
            input_features,
            dtype=object,
        )


# ---------------------------------------------------------
# Separate ordinary continuous values from small counts
# ---------------------------------------------------------

discrete_numeric_features = [
    "Year",
    "recorded_element_count",
    "source_absent_element_count",
    "source_present_missing_element_count",
    "recorded_zero_element_count",
    "recorded_negative_element_count",
    "supply_outcome_recorded_count",
]

clipped_numeric_features = [
    column
    for column in numeric_features
    if column not in discrete_numeric_features
]

assert set(
    clipped_numeric_features
    + discrete_numeric_features
) == set(numeric_features)

# ---------------------------------------------------------
# Create preprocessing pipelines
# ---------------------------------------------------------

continuous_numeric_pipeline = Pipeline(
    steps=[
        (
            "clipper",
            QuantileClipper(
                lower_quantile=0.01,
                upper_quantile=0.99,
            ),
        ),
        (
            "imputer",
            SimpleImputer(
                strategy="median",
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

discrete_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent",
            ),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous_numeric",
            continuous_numeric_pipeline,
            clipped_numeric_features,
        ),
        (
            "discrete_numeric",
            discrete_numeric_pipeline,
            discrete_numeric_features,
        ),
        (
            "binary",
            "passthrough",
            binary_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# ---------------------------------------------------------
# Fit an audit copy using training data only
# ---------------------------------------------------------

preprocessor_audit = clone(
    preprocessor
)

X_train_processed = (
    preprocessor_audit
    .fit_transform(
        X_train,
        y_train,
    )
)

X_validation_processed = (
    preprocessor_audit
    .transform(
        X_validation
    )
)

processed_feature_names = (
    preprocessor_audit
    .get_feature_names_out()
)

# ---------------------------------------------------------
# Audit the processed matrices
# ---------------------------------------------------------

processed_matrix_summary = pd.DataFrame(
    {
        "Dataset": [
            "Training",
            "Validation",
        ],
        "Rows": [
            X_train_processed.shape[0],
            X_validation_processed.shape[0],
        ],
        "Processed columns": [
            X_train_processed.shape[1],
            X_validation_processed.shape[1],
        ],
        "Missing values": [
            np.isnan(
                X_train_processed
            ).sum(),
            np.isnan(
                X_validation_processed
            ).sum(),
        ],
        "Infinite values": [
            np.isinf(
                X_train_processed
            ).sum(),
            np.isinf(
                X_validation_processed
            ).sum(),
        ],
    }
)

print("Processed development-matrix summary:")
display(processed_matrix_summary)

# ---------------------------------------------------------
# Show how categorical columns expanded
# ---------------------------------------------------------

fitted_encoder = (
    preprocessor_audit
    .named_transformers_[
        "categorical"
    ]
    .named_steps[
        "encoder"
    ]
)

categorical_expansion = pd.DataFrame(
    {
        "Categorical feature": (
            categorical_features
        ),
        "Training categories learned": [
            len(categories)
            for categories in (
                fitted_encoder.categories_
            )
        ],
    }
)

print("\nCategorical encoding summary:")
display(categorical_expansion)

# ---------------------------------------------------------
# Inspect training-derived clipping boundaries
# ---------------------------------------------------------

fitted_clipper = (
    preprocessor_audit
    .named_transformers_[
        "continuous_numeric"
    ]
    .named_steps[
        "clipper"
    ]
)

clipping_boundary_table = pd.DataFrame(
    {
        "Feature": (
            clipped_numeric_features
        ),
        "Training lower boundary": (
            fitted_clipper.lower_bounds_
        ),
        "Training upper boundary": (
            fitted_clipper.upper_bounds_
        ),
    }
)

derived_clipping_boundaries = (
    clipping_boundary_table.loc[
        clipping_boundary_table[
            "Feature"
        ].isin(
            [
                "import_share_of_domestic_supply_pct",
                "production_share_of_domestic_supply_pct",
                "export_share_of_domestic_supply_pct",
                "food_share_of_domestic_supply_pct",
                "losses_share_of_domestic_supply_pct",
                "stock_variation_share_of_domestic_supply_pct",
                "production_kg_per_person",
                "imports_kg_per_person",
                "exports_kg_per_person",
                "log_population",
            ]
        )
    ]
    .reset_index(drop=True)
)

print(
    "\nTraining-derived boundaries for "
    "the ten derived features:"
)
display(derived_clipping_boundaries)

# ---------------------------------------------------------
# Inspect selected training-fitted replacement values
# ---------------------------------------------------------

continuous_imputer = (
    preprocessor_audit
    .named_transformers_[
        "continuous_numeric"
    ]
    .named_steps[
        "imputer"
    ]
)

continuous_imputation_table = pd.DataFrame(
    {
        "Feature": (
            clipped_numeric_features
        ),
        "Training median used if missing": (
            continuous_imputer.statistics_
        ),
    }
)

selected_imputation_features = [
    "production_1000t",
    "import_quantity_1000t",
    "export_quantity_1000t",
    "processing_1000t",
    "tourist_consumption_1000t",
    "food_supply_kcal_cap_day_lag1",
    "food_supply_kcal_cap_day_change1",
]

print(
    "\nExamples of training-fitted "
    "missing-value replacements:"
)

display(
    continuous_imputation_table.loc[
        continuous_imputation_table[
            "Feature"
        ].isin(
            selected_imputation_features
        )
    ].reset_index(drop=True)
)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert len(clipped_numeric_features) == 99
assert len(discrete_numeric_features) == 7

assert X_train_processed.shape == (
    2250,
    181,
)

assert X_validation_processed.shape == (
    500,
    181,
)

assert len(processed_feature_names) == 181

assert len(
    set(processed_feature_names)
) == len(processed_feature_names)

assert not np.isnan(
    X_train_processed
).any()

assert not np.isnan(
    X_validation_processed
).any()

assert not np.isinf(
    X_train_processed
).any()

assert not np.isinf(
    X_validation_processed
).any()

assert fitted_clipper.fit_row_count_ == 2250

assert [
    len(categories)
    for categories in (
        fitted_encoder.categories_
    )
] == [
    43,
    8,
    4,
]

print(
    "\nThe preprocessing system was fitted using "
    "training data only and validated successfully."
)

print(
    "The validation data were transformed using "
    "the training-fitted rules."
)

Processed development-matrix summary:


,Dataset,Rows,Processed columns,Missing values,Infinite values
0,Training,2250,181,0,0
1,Validation,500,181,0,0



Categorical encoding summary:


,Categorical feature,Training categories learned
0,Area,43
1,Item Code,8
2,production_status,4



Training-derived boundaries for the ten derived features:


,Feature,Training lower boundary,Training upper boundary
0,import_share_of_domestic_supply_pct,0.0000,161.7115
1,production_share_of_domestic_supply_pct,0.0000,144.8435
2,export_share_of_domestic_supply_pct,0.0000,47.4661
3,food_share_of_domestic_supply_pct,4.2306,100.0000
4,losses_share_of_domestic_supply_pct,0.0000,24.1109
5,stock_variation_share_of_domestic_supply_pct,-39.4000,64.1927
6,production_kg_per_person,0.0000,523.3736
7,imports_kg_per_person,0.0000,283.9973
8,exports_kg_per_person,0.0000,27.2654
9,log_population,11.5584,19.0908



Examples of training-fitted missing-value replacements:


,Feature,Training median used if missing
0,export_quantity_1000t,0.0000
1,import_quantity_1000t,15.0000
2,processing_1000t,6.0000
3,production_1000t,133.0000
4,tourist_consumption_1000t,0.0000
5,food_supply_kcal_cap_day_lag1,124.9100
6,food_supply_kcal_cap_day_change1,0.0800



The preprocessing system was fitted using training data only and validated successfully.
The validation data were transformed using the training-fitted rules.


In [5]:
# ---------------------------------------------------------
# Show the weighting applied by balanced classification
# ---------------------------------------------------------

training_class_counts = (
    y_train
    .value_counts()
    .sort_index()
)

balanced_class_weights = {
    class_value: (
        len(y_train)
        / (
            2
            * class_count
        )
    )
    for class_value, class_count
    in training_class_counts.items()
}

class_weight_table = pd.DataFrame(
    {
        "Outcome": [
            "No shortage",
            "Shortage",
        ],
        "Training observations": [
            training_class_counts[0],
            training_class_counts[1],
        ],
        "Ordinary model weight": [
            1.0,
            1.0,
        ],
        "Balanced model weight": [
            balanced_class_weights[0],
            balanced_class_weights[1],
        ],
    }
)

print("Training-class weighting:")
display(class_weight_table)

# ---------------------------------------------------------
# Define the two logistic-regression models
# ---------------------------------------------------------

logistic_model_definitions = {
    "Ordinary logistic regression": (
        LogisticRegression(
            penalty="l2",
            C=1.0,
            class_weight=None,
            solver="lbfgs",
            max_iter=5000,
            random_state=RANDOM_STATE,
        )
    ),
    "Class-balanced logistic regression": (
        LogisticRegression(
            penalty="l2",
            C=1.0,
            class_weight="balanced",
            solver="lbfgs",
            max_iter=5000,
            random_state=RANDOM_STATE,
        )
    ),
}

fitted_development_models = {}
logistic_validation_rows = []
logistic_validation_predictions = {}
logistic_validation_probabilities = {}

# ---------------------------------------------------------
# Fit using training data and evaluate on validation data
# ---------------------------------------------------------

for model_name, classifier in (
    logistic_model_definitions.items()
):
    model_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                clone(preprocessor),
            ),
            (
                "classifier",
                classifier,
            ),
        ]
    )

    model_pipeline.fit(
        X_train,
        y_train,
    )

    validation_probability = (
        model_pipeline.predict_proba(
            X_validation
        )[:, 1]
    )

    validation_result, (
        validation_prediction
    ) = evaluate_probabilities(
        y_true=y_validation,
        predicted_probability=(
            validation_probability
        ),
        threshold=0.50,
        model_name=model_name,
        dataset_name="Validation",
    )

    fitted_development_models[
        model_name
    ] = model_pipeline

    logistic_validation_probabilities[
        model_name
    ] = validation_probability

    logistic_validation_predictions[
        model_name
    ] = validation_prediction

    logistic_validation_rows.append(
        validation_result
    )

    validation_evaluation_rows.append(
        validation_result
    )

# ---------------------------------------------------------
# Compare the models with the no-shortage baseline
# ---------------------------------------------------------

validation_model_comparison = pd.DataFrame(
    validation_evaluation_rows
)

comparison_columns = [
    "Model",
    "Threshold",
    "Predicted events",
    "True positives",
    "False positives",
    "False negatives",
    "Accuracy",
    "Balanced accuracy",
    "Precision",
    "Recall",
    "F1",
    "F2",
    "ROC-AUC",
    "PR-AUC",
    "Brier score",
]

print("\nValidation model comparison:")
display(
    validation_model_comparison[
        comparison_columns
    ]
)

# ---------------------------------------------------------
# Display confusion matrices in ordinary language
# ---------------------------------------------------------

logistic_confusion_rows = []

for result in logistic_validation_rows:
    logistic_confusion_rows.append(
        {
            "Model": result["Model"],
            "Actual shortages": (
                result["Actual events"]
            ),
            "Shortages detected": (
                result["True positives"]
            ),
            "Shortages missed": (
                result["False negatives"]
            ),
            "False warnings": (
                result["False positives"]
            ),
            "Correct non-shortages": (
                result["True negatives"]
            ),
            "Warnings issued": (
                result["Predicted events"]
            ),
        }
    )

logistic_confusion_summary = pd.DataFrame(
    logistic_confusion_rows
)

print("\nLogistic-regression warning summary:")
display(logistic_confusion_summary)

# ---------------------------------------------------------
# Audit probability behaviour
# ---------------------------------------------------------

probability_summary_rows = []

for model_name, probabilities in (
    logistic_validation_probabilities.items()
):
    probability_summary_rows.append(
        {
            "Model": model_name,
            "Minimum probability": (
                probabilities.min()
            ),
            "Median probability": (
                np.median(probabilities)
            ),
            "Mean probability": (
                probabilities.mean()
            ),
            "95th percentile probability": (
                np.quantile(
                    probabilities,
                    0.95,
                )
            ),
            "Maximum probability": (
                probabilities.max()
            ),
        }
    )

probability_summary = pd.DataFrame(
    probability_summary_rows
)

print("\nValidation predicted-probability summary:")
display(probability_summary)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert len(
    fitted_development_models
) == 2

assert len(
    logistic_validation_rows
) == 2

for model_name, model_pipeline in (
    fitted_development_models.items()
):
    probabilities = (
        logistic_validation_probabilities[
            model_name
        ]
    )

    assert len(probabilities) == 500
    assert np.isfinite(
        probabilities
    ).all()

    assert (
        probabilities >= 0
    ).all()

    assert (
        probabilities <= 1
    ).all()

    fitted_pipeline_preprocessor = (
        model_pipeline
        .named_steps[
            "preprocessor"
        ]
    )

    fitted_pipeline_clipper = (
        fitted_pipeline_preprocessor
        .named_transformers_[
            "continuous_numeric"
        ]
        .named_steps[
            "clipper"
        ]
    )

    assert (
        fitted_pipeline_clipper
        .fit_row_count_
        == 2250
    )

assert len(
    validation_evaluation_rows
) == 3

print(
    "\nBoth logistic-regression baselines were "
    "trained on the training period and evaluated "
    "on the validation period successfully."
)

print(
    "The locked test period was not used."
)

Training-class weighting:


,Outcome,Training observations,Ordinary model weight,Balanced model weight
0,No shortage,2019,1.0000,0.5572
1,Shortage,231,1.0000,4.8701



Validation model comparison:


/opt/anaconda3/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,Model,Threshold,Predicted events,True positives,False positives,False negatives,Accuracy,Balanced accuracy,Precision,Recall,F1,F2,ROC-AUC,PR-AUC,Brier score
0,Always predict no shortage,0.5000,0,0,0,52,0.8960,0.5000,0.0000,0.0000,0.0000,0.0000,0.5000,0.1040,0.0932
1,Ordinary logistic regression,0.5000,17,8,9,44,0.8940,0.5669,0.4706,0.1538,0.2319,0.1778,0.7172,0.3225,0.0859
2,Class-balanced logistic regression,0.5000,199,36,163,16,0.6420,0.6642,0.1809,0.6923,0.2869,0.4423,0.7068,0.3104,0.2186



Logistic-regression warning summary:


,Model,Actual shortages,Shortages detected,Shortages missed,False warnings,Correct non-shortages,Warnings issued
0,Ordinary logistic regression,52,8,44,9,439,17
1,Class-balanced logistic regression,52,36,16,163,285,199



Validation predicted-probability summary:


,Model,Minimum probability,Median probability,Mean probability,95th percentile probability,Maximum probability
0,Ordinary logistic regression,0.0000,0.0738,0.1205,0.4095,0.8056
1,Class-balanced logistic regression,0.0000,0.3675,0.3948,0.8585,0.9803



Both logistic-regression baselines were trained on the training period and evaluated on the validation period successfully.
The locked test period was not used.


In [6]:
# ---------------------------------------------------------
# Evaluate a range of probability thresholds
# ---------------------------------------------------------

threshold_grid = np.round(
    np.arange(
        0.01,
        1.00,
        0.01,
    ),
    2,
)

threshold_evaluation_rows = []

for model_name, probabilities in (
    logistic_validation_probabilities.items()
):
    for threshold in threshold_grid:
        threshold_result, _ = (
            evaluate_probabilities(
                y_true=y_validation,
                predicted_probability=(
                    probabilities
                ),
                threshold=threshold,
                model_name=model_name,
                dataset_name="Validation",
            )
        )

        threshold_evaluation_rows.append(
            threshold_result
        )

logistic_threshold_results = pd.DataFrame(
    threshold_evaluation_rows
)

# ---------------------------------------------------------
# Identify the best validation threshold for F1 and F2
# ---------------------------------------------------------

best_threshold_rows = []
selected_f2_thresholds = {}

for model_name in (
    logistic_validation_probabilities
):
    model_thresholds = (
        logistic_threshold_results.loc[
            logistic_threshold_results[
                "Model"
            ].eq(model_name)
        ]
        .copy()
    )

    best_f1_row = (
        model_thresholds
        .sort_values(
            [
                "F1",
                "Precision",
                "Recall",
                "Threshold",
            ],
            ascending=[
                False,
                False,
                False,
                False,
            ],
        )
        .iloc[0]
    )

    best_f2_row = (
        model_thresholds
        .sort_values(
            [
                "F2",
                "Recall",
                "Precision",
                "Threshold",
            ],
            ascending=[
                False,
                False,
                False,
                False,
            ],
        )
        .iloc[0]
    )

    selected_f2_thresholds[
        model_name
    ] = float(
        best_f2_row["Threshold"]
    )

    for selection_name, row in [
        (
            "Highest validation F1",
            best_f1_row,
        ),
        (
            "Highest validation F2",
            best_f2_row,
        ),
    ]:
        best_threshold_rows.append(
            {
                "Model": model_name,
                "Selection rule": (
                    selection_name
                ),
                "Threshold": row["Threshold"],
                "Warnings issued": (
                    row["Predicted events"]
                ),
                "Shortages detected": (
                    row["True positives"]
                ),
                "Shortages missed": (
                    row["False negatives"]
                ),
                "False warnings": (
                    row["False positives"]
                ),
                "Precision": row["Precision"],
                "Recall": row["Recall"],
                "F1": row["F1"],
                "F2": row["F2"],
                "Balanced accuracy": (
                    row["Balanced accuracy"]
                ),
            }
        )

best_logistic_thresholds = pd.DataFrame(
    best_threshold_rows
)

print("Best validation thresholds by scoring priority:")
display(best_logistic_thresholds)

# ---------------------------------------------------------
# Compare default and F2-selected thresholds directly
# ---------------------------------------------------------

default_and_selected_rows = []

for model_name, probabilities in (
    logistic_validation_probabilities.items()
):
    for threshold_label, threshold in [
        (
            "Default 0.50 threshold",
            0.50,
        ),
        (
            "Validation-selected F2 threshold",
            selected_f2_thresholds[
                model_name
            ],
        ),
    ]:
        result, _ = evaluate_probabilities(
            y_true=y_validation,
            predicted_probability=(
                probabilities
            ),
            threshold=threshold,
            model_name=model_name,
            dataset_name="Validation",
        )

        default_and_selected_rows.append(
            {
                "Model": model_name,
                "Threshold rule": (
                    threshold_label
                ),
                "Threshold": threshold,
                "Warnings issued": (
                    result["Predicted events"]
                ),
                "Shortages detected": (
                    result["True positives"]
                ),
                "Shortages missed": (
                    result["False negatives"]
                ),
                "False warnings": (
                    result["False positives"]
                ),
                "Precision": (
                    result["Precision"]
                ),
                "Recall": result["Recall"],
                "F1": result["F1"],
                "F2": result["F2"],
                "Balanced accuracy": (
                    result[
                        "Balanced accuracy"
                    ]
                ),
                "PR-AUC": result["PR-AUC"],
                "Brier score": (
                    result["Brier score"]
                ),
            }
        )

default_and_selected_comparison = (
    pd.DataFrame(
        default_and_selected_rows
    )
)

print(
    "\nDefault versus F2-selected "
    "validation thresholds:"
)
display(default_and_selected_comparison)

# ---------------------------------------------------------
# Show operational trade-offs for the ordinary model
# ---------------------------------------------------------

ordinary_model_name = (
    "Ordinary logistic regression"
)

ordinary_threshold_tradeoff = (
    logistic_threshold_results.loc[
        logistic_threshold_results[
            "Model"
        ].eq(ordinary_model_name)
        & logistic_threshold_results[
            "Threshold"
        ].isin(
            [
                0.10,
                0.15,
                0.20,
                0.25,
                0.30,
                0.40,
                0.50,
            ]
        ),
        [
            "Threshold",
            "Predicted events",
            "True positives",
            "False positives",
            "False negatives",
            "Precision",
            "Recall",
            "F1",
            "F2",
            "Balanced accuracy",
        ],
    ]
    .rename(
        columns={
            "Predicted events": (
                "Warnings issued"
            ),
            "True positives": (
                "Shortages detected"
            ),
            "False positives": (
                "False warnings"
            ),
            "False negatives": (
                "Shortages missed"
            ),
        }
    )
    .reset_index(drop=True)
)

print(
    "\nOrdinary logistic-regression "
    "threshold trade-off:"
)
display(ordinary_threshold_tradeoff)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert len(
    logistic_threshold_results
) == (
    2 * len(threshold_grid)
)

assert set(
    selected_f2_thresholds
) == set(
    logistic_validation_probabilities
)

for model_name, threshold in (
    selected_f2_thresholds.items()
):
    assert 0.01 <= threshold <= 0.99

    selected_row = (
        logistic_threshold_results.loc[
            logistic_threshold_results[
                "Model"
            ].eq(model_name)
            & logistic_threshold_results[
                "Threshold"
            ].eq(threshold)
        ]
    )

    assert len(selected_row) == 1

assert len(
    best_logistic_thresholds
) == 4

assert len(
    default_and_selected_comparison
) == 4

print(
    "\nLogistic-regression threshold sensitivity "
    "was evaluated successfully using validation "
    "data only."
)

print(
    "No final threshold has been applied to the "
    "locked test period."
)

Best validation thresholds by scoring priority:


,Model,Selection rule,Threshold,Warnings issued,Shortages detected,Shortages missed,False warnings,Precision,Recall,F1,F2,Balanced accuracy
0,Ordinary logistic regression,Highest validation F1,0.3100,44,19,33,25,0.4318,0.3654,0.3958,0.3770,0.6548
1,Ordinary logistic regression,Highest validation F2,0.1300,157,36,16,121,0.2293,0.6923,0.3445,0.4932,0.7111
2,Class-balanced logistic regression,Highest validation F1,0.8200,41,18,34,23,0.4390,0.3462,0.3871,0.3614,0.6474
3,Class-balanced logistic regression,Highest validation F2,0.5700,164,35,17,129,0.2134,0.6731,0.3241,0.4704,0.6926



Default versus F2-selected validation thresholds:


,Model,Threshold rule,Threshold,Warnings issued,Shortages detected,Shortages missed,False warnings,Precision,Recall,F1,F2,Balanced accuracy,PR-AUC,Brier score
0,Ordinary logistic regression,Default 0.50 threshold,0.5000,17,8,44,9,0.4706,0.1538,0.2319,0.1778,0.5669,0.3225,0.0859
1,Ordinary logistic regression,Validation-selected F2 threshold,0.1300,157,36,16,121,0.2293,0.6923,0.3445,0.4932,0.7111,0.3225,0.0859
2,Class-balanced logistic regression,Default 0.50 threshold,0.5000,199,36,16,163,0.1809,0.6923,0.2869,0.4423,0.6642,0.3104,0.2186
3,Class-balanced logistic regression,Validation-selected F2 threshold,0.5700,164,35,17,129,0.2134,0.6731,0.3241,0.4704,0.6926,0.3104,0.2186



Ordinary logistic-regression threshold trade-off:


,Threshold,Warnings issued,Shortages detected,False warnings,Shortages missed,Precision,Recall,F1,F2,Balanced accuracy
0,0.1000,191,36,155,16,0.1885,0.6923,0.2963,0.4511,0.6732
1,0.1500,140,33,107,19,0.2357,0.6346,0.3438,0.4741,0.6979
2,0.2000,96,23,73,29,0.2396,0.4423,0.3108,0.3783,0.6397
3,0.2500,66,19,47,33,0.2879,0.3654,0.3220,0.3467,0.6302
4,0.3000,46,19,27,33,0.4130,0.3654,0.3878,0.3740,0.6526
5,0.4000,30,14,16,38,0.4667,0.2692,0.3415,0.2941,0.6168
6,0.5000,17,8,9,44,0.4706,0.1538,0.2319,0.1778,0.5669



Logistic-regression threshold sensitivity was evaluated successfully using validation data only.
No final threshold has been applied to the locked test period.


In [7]:
# ---------------------------------------------------------
# Define conservative random-forest baselines
# ---------------------------------------------------------

random_forest_definitions = {
    "Ordinary random forest": (
        RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=3,
            max_features="sqrt",
            class_weight=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    ),
    "Class-balanced random forest": (
        RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=3,
            max_features="sqrt",
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    ),
}

forest_validation_probabilities = {}
forest_validation_rows = []

# ---------------------------------------------------------
# Train using the training period only
# ---------------------------------------------------------

for model_name, classifier in (
    random_forest_definitions.items()
):
    forest_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                clone(preprocessor),
            ),
            (
                "classifier",
                classifier,
            ),
        ]
    )

    forest_pipeline.fit(
        X_train,
        y_train,
    )

    validation_probability = (
        forest_pipeline.predict_proba(
            X_validation
        )[:, 1]
    )

    validation_result, _ = (
        evaluate_probabilities(
            y_true=y_validation,
            predicted_probability=(
                validation_probability
            ),
            threshold=0.50,
            model_name=model_name,
            dataset_name="Validation",
        )
    )

    fitted_development_models[
        model_name
    ] = forest_pipeline

    forest_validation_probabilities[
        model_name
    ] = validation_probability

    forest_validation_rows.append(
        validation_result
    )

    validation_evaluation_rows.append(
        validation_result
    )

# ---------------------------------------------------------
# Display default-threshold performance
# ---------------------------------------------------------

forest_default_comparison = pd.DataFrame(
    forest_validation_rows
)

print(
    "Random-forest validation performance "
    "at the default 0.50 threshold:"
)

display(
    forest_default_comparison[
        [
            "Model",
            "Threshold",
            "Predicted events",
            "True positives",
            "False positives",
            "False negatives",
            "Accuracy",
            "Balanced accuracy",
            "Precision",
            "Recall",
            "F1",
            "F2",
            "ROC-AUC",
            "PR-AUC",
            "Brier score",
        ]
    ]
)

# ---------------------------------------------------------
# Select F2-oriented validation thresholds
# ---------------------------------------------------------

forest_threshold_rows = []

for model_name, probabilities in (
    forest_validation_probabilities.items()
):
    for threshold in threshold_grid:
        result, _ = evaluate_probabilities(
            y_true=y_validation,
            predicted_probability=(
                probabilities
            ),
            threshold=threshold,
            model_name=model_name,
            dataset_name="Validation",
        )

        forest_threshold_rows.append(
            result
        )

forest_threshold_results = pd.DataFrame(
    forest_threshold_rows
)

forest_selected_threshold_rows = []

for model_name in (
    forest_validation_probabilities
):
    model_thresholds = (
        forest_threshold_results.loc[
            forest_threshold_results[
                "Model"
            ].eq(model_name)
        ]
        .copy()
    )

    best_f2_row = (
        model_thresholds
        .sort_values(
            [
                "F2",
                "Recall",
                "Precision",
                "Threshold",
            ],
            ascending=[
                False,
                False,
                False,
                False,
            ],
        )
        .iloc[0]
    )

    selected_f2_thresholds[
        model_name
    ] = float(
        best_f2_row["Threshold"]
    )

    forest_selected_threshold_rows.append(
        {
            "Model": model_name,
            "Selected threshold": (
                best_f2_row["Threshold"]
            ),
            "Warnings issued": (
                best_f2_row[
                    "Predicted events"
                ]
            ),
            "Shortages detected": (
                best_f2_row[
                    "True positives"
                ]
            ),
            "Shortages missed": (
                best_f2_row[
                    "False negatives"
                ]
            ),
            "False warnings": (
                best_f2_row[
                    "False positives"
                ]
            ),
            "Precision": (
                best_f2_row["Precision"]
            ),
            "Recall": (
                best_f2_row["Recall"]
            ),
            "F1": best_f2_row["F1"],
            "F2": best_f2_row["F2"],
            "Balanced accuracy": (
                best_f2_row[
                    "Balanced accuracy"
                ]
            ),
            "ROC-AUC": (
                best_f2_row["ROC-AUC"]
            ),
            "PR-AUC": (
                best_f2_row["PR-AUC"]
            ),
            "Brier score": (
                best_f2_row[
                    "Brier score"
                ]
            ),
        }
    )

forest_selected_thresholds = pd.DataFrame(
    forest_selected_threshold_rows
)

print(
    "\nRandom-forest F2-selected "
    "validation thresholds:"
)
display(forest_selected_thresholds)

# ---------------------------------------------------------
# Compare all four trained models fairly
# ---------------------------------------------------------

all_validation_probabilities = {
    **logistic_validation_probabilities,
    **forest_validation_probabilities,
}

selected_model_comparison_rows = []

for model_name, probabilities in (
    all_validation_probabilities.items()
):
    selected_threshold = (
        selected_f2_thresholds[
            model_name
        ]
    )

    selected_result, _ = (
        evaluate_probabilities(
            y_true=y_validation,
            predicted_probability=(
                probabilities
            ),
            threshold=selected_threshold,
            model_name=model_name,
            dataset_name="Validation",
        )
    )

    selected_model_comparison_rows.append(
        {
            "Model": model_name,
            "Selected threshold": (
                selected_threshold
            ),
            "Warnings issued": (
                selected_result[
                    "Predicted events"
                ]
            ),
            "Shortages detected": (
                selected_result[
                    "True positives"
                ]
            ),
            "Shortages missed": (
                selected_result[
                    "False negatives"
                ]
            ),
            "False warnings": (
                selected_result[
                    "False positives"
                ]
            ),
            "Precision": (
                selected_result[
                    "Precision"
                ]
            ),
            "Recall": (
                selected_result[
                    "Recall"
                ]
            ),
            "F1": selected_result["F1"],
            "F2": selected_result["F2"],
            "Balanced accuracy": (
                selected_result[
                    "Balanced accuracy"
                ]
            ),
            "ROC-AUC": (
                selected_result["ROC-AUC"]
            ),
            "PR-AUC": (
                selected_result["PR-AUC"]
            ),
            "Brier score": (
                selected_result[
                    "Brier score"
                ]
            ),
        }
    )

selected_model_comparison = (
    pd.DataFrame(
        selected_model_comparison_rows
    )
    .sort_values(
        [
            "F2",
            "PR-AUC",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    "\nAll models at their F2-selected "
    "validation thresholds:"
)
display(selected_model_comparison)

# ---------------------------------------------------------
# Compare threshold-independent ranking and probability quality
# ---------------------------------------------------------

ranking_comparison = (
    selected_model_comparison[
        [
            "Model",
            "ROC-AUC",
            "PR-AUC",
            "Brier score",
        ]
    ]
    .sort_values(
        "PR-AUC",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    "\nThreshold-independent validation comparison:"
)
display(ranking_comparison)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert len(
    forest_validation_probabilities
) == 2

assert len(
    fitted_development_models
) == 4

assert len(
    forest_threshold_results
) == (
    2 * len(threshold_grid)
)

assert len(
    selected_model_comparison
) == 4

assert set(
    selected_model_comparison[
        "Model"
    ]
) == set(
    all_validation_probabilities
)

for model_name, probabilities in (
    forest_validation_probabilities.items()
):
    assert len(probabilities) == 500
    assert np.isfinite(
        probabilities
    ).all()

    assert (
        probabilities >= 0
    ).all()

    assert (
        probabilities <= 1
    ).all()

    assert (
        model_name
        in selected_f2_thresholds
    )

print(
    "\nThe random-forest baselines were trained "
    "and evaluated successfully."
)

print(
    "All model and threshold comparisons used "
    "validation data only. The test period remains locked."
)

Random-forest validation performance at the default 0.50 threshold:


,Model,Threshold,Predicted events,True positives,False positives,False negatives,Accuracy,Balanced accuracy,Precision,Recall,F1,F2,ROC-AUC,PR-AUC,Brier score
0,Ordinary random forest,0.5000,18,8,10,44,0.8920,0.5658,0.4444,0.1538,0.2286,0.1770,0.7395,0.2835,0.0869
1,Class-balanced random forest,0.5000,26,11,15,41,0.8880,0.5890,0.4231,0.2115,0.2821,0.2350,0.7596,0.2902,0.0905



Random-forest F2-selected validation thresholds:


,Model,Selected threshold,Warnings issued,Shortages detected,Shortages missed,False warnings,Precision,Recall,F1,F2,Balanced accuracy,ROC-AUC,PR-AUC,Brier score
0,Ordinary random forest,0.1200,189,37,15,152,0.1958,0.7115,0.3071,0.4660,0.6861,0.7395,0.2835,0.0869
1,Class-balanced random forest,0.1700,175,36,16,139,0.2057,0.6923,0.3172,0.4700,0.6910,0.7596,0.2902,0.0905



All models at their F2-selected validation thresholds:


,Model,Selected threshold,Warnings issued,Shortages detected,Shortages missed,False warnings,Precision,Recall,F1,F2,Balanced accuracy,ROC-AUC,PR-AUC,Brier score
0,Ordinary logistic regression,0.1300,157,36,16,121,0.2293,0.6923,0.3445,0.4932,0.7111,0.7172,0.3225,0.0859
1,Class-balanced logistic regression,0.5700,164,35,17,129,0.2134,0.6731,0.3241,0.4704,0.6926,0.7068,0.3104,0.2186
2,Class-balanced random forest,0.1700,175,36,16,139,0.2057,0.6923,0.3172,0.4700,0.6910,0.7596,0.2902,0.0905
3,Ordinary random forest,0.1200,189,37,15,152,0.1958,0.7115,0.3071,0.4660,0.6861,0.7395,0.2835,0.0869



Threshold-independent validation comparison:


,Model,ROC-AUC,PR-AUC,Brier score
0,Ordinary logistic regression,0.7172,0.3225,0.0859
1,Class-balanced logistic regression,0.7068,0.3104,0.2186
2,Class-balanced random forest,0.7596,0.2902,0.0905
3,Ordinary random forest,0.7395,0.2835,0.0869



The random-forest baselines were trained and evaluated successfully.
All model and threshold comparisons used validation data only. The test period remains locked.


In [8]:
# ---------------------------------------------------------
# Define expanding temporal validation folds
# ---------------------------------------------------------

temporal_cv_definitions = [
    {
        "Fold": 1,
        "Training year start": 2010,
        "Training year end": 2014,
        "Validation year": 2015,
    },
    {
        "Fold": 2,
        "Training year start": 2010,
        "Training year end": 2015,
        "Validation year": 2016,
    },
    {
        "Fold": 3,
        "Training year start": 2010,
        "Training year end": 2016,
        "Validation year": 2017,
    },
    {
        "Fold": 4,
        "Training year start": 2010,
        "Training year end": 2017,
        "Validation year": 2018,
    },
]

temporal_cv_table = pd.DataFrame(
    temporal_cv_definitions
)

# Add fold sizes and event rates
fold_size_rows = []

for fold_definition in temporal_cv_definitions:
    train_end_year = (
        fold_definition[
            "Training year end"
        ]
    )

    validation_year = (
        fold_definition[
            "Validation year"
        ]
    )

    fold_train = train_data.loc[
        train_data["Year"]
        .between(
            2010,
            train_end_year,
        )
    ]

    fold_validation = train_data.loc[
        train_data["Year"]
        .eq(validation_year)
    ]

    fold_size_rows.append(
        {
            "Fold": (
                fold_definition["Fold"]
            ),
            "Training predictor years": (
                f"2010–{train_end_year}"
            ),
            "Validation predictor year": (
                validation_year
            ),
            "Training rows": (
                len(fold_train)
            ),
            "Training events": (
                fold_train[
                    target_column
                ].sum()
            ),
            "Validation rows": (
                len(fold_validation)
            ),
            "Validation events": (
                fold_validation[
                    target_column
                ].sum()
            ),
            "Validation event rate %": (
                100
                * fold_validation[
                    target_column
                ].mean()
            ),
        }
    )

temporal_cv_fold_summary = pd.DataFrame(
    fold_size_rows
)

print("Expanding-window temporal fold summary:")
display(temporal_cv_fold_summary)

# ---------------------------------------------------------
# Recreate model definitions without deprecated syntax
# ---------------------------------------------------------

temporal_cv_model_definitions = {
    "Ordinary logistic regression": (
        LogisticRegression(
            C=1.0,
            l1_ratio=0,
            class_weight=None,
            solver="lbfgs",
            max_iter=5000,
            random_state=RANDOM_STATE,
        )
    ),
    "Class-balanced logistic regression": (
        LogisticRegression(
            C=1.0,
            l1_ratio=0,
            class_weight="balanced",
            solver="lbfgs",
            max_iter=5000,
            random_state=RANDOM_STATE,
        )
    ),
    "Ordinary random forest": (
        RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=3,
            max_features="sqrt",
            class_weight=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    ),
    "Class-balanced random forest": (
        RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=3,
            max_features="sqrt",
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    ),
}

# ---------------------------------------------------------
# Run the expanding-window evaluation
# ---------------------------------------------------------

temporal_cv_result_rows = []

for fold_definition in temporal_cv_definitions:
    fold_number = (
        fold_definition["Fold"]
    )

    train_start_year = (
        fold_definition[
            "Training year start"
        ]
    )

    train_end_year = (
        fold_definition[
            "Training year end"
        ]
    )

    validation_year = (
        fold_definition[
            "Validation year"
        ]
    )

    fold_train_mask = (
        train_data["Year"]
        .between(
            train_start_year,
            train_end_year,
        )
    )

    fold_validation_mask = (
        train_data["Year"]
        .eq(validation_year)
    )

    X_fold_train = train_data.loc[
        fold_train_mask,
        feature_columns,
    ]

    y_fold_train = train_data.loc[
        fold_train_mask,
        target_column,
    ].astype("int8")

    X_fold_validation = train_data.loc[
        fold_validation_mask,
        feature_columns,
    ]

    y_fold_validation = train_data.loc[
        fold_validation_mask,
        target_column,
    ].astype("int8")

    fold_event_rate = (
        y_fold_validation.mean()
    )

    for model_name, classifier in (
        temporal_cv_model_definitions.items()
    ):
        fold_pipeline = Pipeline(
            steps=[
                (
                    "preprocessor",
                    clone(preprocessor),
                ),
                (
                    "classifier",
                    clone(classifier),
                ),
            ]
        )

        fold_pipeline.fit(
            X_fold_train,
            y_fold_train,
        )

        fold_probability = (
            fold_pipeline.predict_proba(
                X_fold_validation
            )[:, 1]
        )

        fold_pr_auc = (
            average_precision_score(
                y_fold_validation,
                fold_probability,
            )
        )

        fold_roc_auc = roc_auc_score(
            y_fold_validation,
            fold_probability,
        )

        fold_brier = brier_score_loss(
            y_fold_validation,
            fold_probability,
        )

        temporal_cv_result_rows.append(
            {
                "Fold": fold_number,
                "Model": model_name,
                "Training year end": (
                    train_end_year
                ),
                "Validation year": (
                    validation_year
                ),
                "Training rows": (
                    len(X_fold_train)
                ),
                "Validation rows": (
                    len(X_fold_validation)
                ),
                "Validation events": (
                    y_fold_validation.sum()
                ),
                "Validation event rate": (
                    fold_event_rate
                ),
                "ROC-AUC": fold_roc_auc,
                "PR-AUC": fold_pr_auc,
                "PR-AUC baseline": (
                    fold_event_rate
                ),
                "PR-AUC lift over baseline": (
                    fold_pr_auc
                    / fold_event_rate
                ),
                "Brier score": fold_brier,
            }
        )

        fitted_fold_clipper = (
            fold_pipeline
            .named_steps[
                "preprocessor"
            ]
            .named_transformers_[
                "continuous_numeric"
            ]
            .named_steps[
                "clipper"
            ]
        )

        assert (
            fitted_fold_clipper
            .fit_row_count_
            == len(X_fold_train)
        )

temporal_cv_results = pd.DataFrame(
    temporal_cv_result_rows
)

# ---------------------------------------------------------
# Display year-by-year PR-AUC behaviour
# ---------------------------------------------------------

temporal_pr_auc_pivot = (
    temporal_cv_results
    .pivot(
        index="Model",
        columns="Validation year",
        values="PR-AUC",
    )
    .reset_index()
)

print("\nTemporal cross-validation PR-AUC by year:")
display(temporal_pr_auc_pivot)

# ---------------------------------------------------------
# Aggregate temporal stability
# ---------------------------------------------------------

temporal_cv_summary = (
    temporal_cv_results
    .groupby(
        "Model",
        observed=True,
    )
    .agg(
        Mean_ROC_AUC=(
            "ROC-AUC",
            "mean",
        ),
        Minimum_ROC_AUC=(
            "ROC-AUC",
            "min",
        ),
        Mean_PR_AUC=(
            "PR-AUC",
            "mean",
        ),
        Minimum_PR_AUC=(
            "PR-AUC",
            "min",
        ),
        Maximum_PR_AUC=(
            "PR-AUC",
            "max",
        ),
        PR_AUC_standard_deviation=(
            "PR-AUC",
            "std",
        ),
        Mean_PR_AUC_lift=(
            "PR-AUC lift over baseline",
            "mean",
        ),
        Mean_Brier_score=(
            "Brier score",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "Mean_PR_AUC",
            "Mean_Brier_score",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

print(
    "\nExpanding-window temporal "
    "cross-validation summary:"
)
display(temporal_cv_summary)

# ---------------------------------------------------------
# Compare cross-validation with 2019–2020 validation
# ---------------------------------------------------------

development_stability_comparison = (
    temporal_cv_summary[
        [
            "Model",
            "Mean_ROC_AUC",
            "Mean_PR_AUC",
            "Minimum_PR_AUC",
            "PR_AUC_standard_deviation",
            "Mean_Brier_score",
        ]
    ]
    .merge(
        ranking_comparison.rename(
            columns={
                "ROC-AUC": (
                    "2019–2020 ROC-AUC"
                ),
                "PR-AUC": (
                    "2019–2020 PR-AUC"
                ),
                "Brier score": (
                    "2019–2020 Brier score"
                ),
            }
        ),
        on="Model",
        how="left",
        validate="one_to_one",
    )
)

print(
    "\nEarlier-year stability compared with "
    "the main validation period:"
)
display(development_stability_comparison)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert len(
    temporal_cv_results
) == 16

assert temporal_cv_results[
    "Validation year"
].isin(
    [2015, 2016, 2017, 2018]
).all()

assert temporal_cv_results[
    "Training year end"
].max() == 2017

assert (
    temporal_cv_results[
        "Validation year"
    ]
    > temporal_cv_results[
        "Training year end"
    ]
).all()

assert temporal_cv_results[
    "PR-AUC"
].notna().all()

assert temporal_cv_results[
    "ROC-AUC"
].notna().all()

assert temporal_cv_results[
    "Brier score"
].notna().all()

assert (
    temporal_cv_results[
        "PR-AUC lift over baseline"
    ]
    > 0
).all()

assert len(
    temporal_cv_summary
) == 4

print(
    "\nExpanding-window temporal "
    "cross-validation completed successfully."
)

print(
    "Only predictor years up to 2018 were used. "
    "The 2019–2020 validation and locked test "
    "periods were not involved in these folds."
)

Expanding-window temporal fold summary:


,Fold,Training predictor years,Validation predictor year,Training rows,Training events,Validation rows,Validation events,Validation event rate %
0,1,2010–2014,2015,1253,128,249,29,11.6466
1,2,2010–2015,2016,1502,157,249,14,5.6225
2,3,2010–2016,2017,1751,171,249,26,10.4418
3,4,2010–2017,2018,2000,197,250,34,13.6000



Temporal cross-validation PR-AUC by year:


Validation year,Model,2015,2016,2017,2018
0,Class-balanced logistic regression,0.2478,0.1704,0.3176,0.2199
1,Class-balanced random forest,0.2663,0.1621,0.4209,0.2449
2,Ordinary logistic regression,0.2908,0.1395,0.3004,0.2250
3,Ordinary random forest,0.3259,0.1980,0.4990,0.3039



Expanding-window temporal cross-validation summary:


,Model,Mean_ROC_AUC,Minimum_ROC_AUC,Mean_PR_AUC,Minimum_PR_AUC,Maximum_PR_AUC,PR_AUC_standard_deviation,Mean_PR_AUC_lift,Mean_Brier_score
0,Ordinary random forest,0.7314,0.6739,0.3317,0.1980,0.4990,0.1248,3.3332,0.0847
1,Class-balanced random forest,0.7284,0.6702,0.2735,0.1621,0.4209,0.1080,2.7501,0.0909
2,Class-balanced logistic regression,0.6872,0.6077,0.2389,0.1704,0.3176,0.0614,2.4539,0.1967
3,Ordinary logistic regression,0.6933,0.6145,0.2389,0.1395,0.3004,0.0743,2.3770,0.0943



Earlier-year stability compared with the main validation period:


,Model,Mean_ROC_AUC,Mean_PR_AUC,Minimum_PR_AUC,PR_AUC_standard_deviation,Mean_Brier_score,2019–2020 ROC-AUC,2019–2020 PR-AUC,2019–2020 Brier score
0,Ordinary random forest,0.7314,0.3317,0.1980,0.1248,0.0847,0.7395,0.2835,0.0869
1,Class-balanced random forest,0.7284,0.2735,0.1621,0.1080,0.0909,0.7596,0.2902,0.0905
2,Class-balanced logistic regression,0.6872,0.2389,0.1704,0.0614,0.1967,0.7068,0.3104,0.2186
3,Ordinary logistic regression,0.6933,0.2389,0.1395,0.0743,0.0943,0.7172,0.3225,0.0859



Expanding-window temporal cross-validation completed successfully.
Only predictor years up to 2018 were used. The 2019–2020 validation and locked test periods were not involved in these folds.


In [9]:
# ---------------------------------------------------------
# Define a small, controlled random-forest search
# ---------------------------------------------------------

random_forest_tuning_configurations = [
    {
        "Configuration": "RF_A",
        "Maximum depth": None,
        "Minimum leaf size": 3,
    },
    {
        "Configuration": "RF_B",
        "Maximum depth": 12,
        "Minimum leaf size": 3,
    },
    {
        "Configuration": "RF_C",
        "Maximum depth": 8,
        "Minimum leaf size": 3,
    },
    {
        "Configuration": "RF_D",
        "Maximum depth": None,
        "Minimum leaf size": 6,
    },
    {
        "Configuration": "RF_E",
        "Maximum depth": 12,
        "Minimum leaf size": 6,
    },
    {
        "Configuration": "RF_F",
        "Maximum depth": 8,
        "Minimum leaf size": 6,
    },
]

rf_tuning_result_rows = []

# ---------------------------------------------------------
# Evaluate each configuration on every temporal fold
# ---------------------------------------------------------

for configuration in (
    random_forest_tuning_configurations
):
    configuration_name = (
        configuration["Configuration"]
    )

    maximum_depth = (
        configuration["Maximum depth"]
    )

    minimum_leaf_size = (
        configuration[
            "Minimum leaf size"
        ]
    )

    for fold_definition in (
        temporal_cv_definitions
    ):
        fold_number = (
            fold_definition["Fold"]
        )

        train_start_year = (
            fold_definition[
                "Training year start"
            ]
        )

        train_end_year = (
            fold_definition[
                "Training year end"
            ]
        )

        validation_year = (
            fold_definition[
                "Validation year"
            ]
        )

        fold_train_mask = (
            train_data["Year"]
            .between(
                train_start_year,
                train_end_year,
            )
        )

        fold_validation_mask = (
            train_data["Year"]
            .eq(validation_year)
        )

        X_fold_train = train_data.loc[
            fold_train_mask,
            feature_columns,
        ]

        y_fold_train = train_data.loc[
            fold_train_mask,
            target_column,
        ].astype("int8")

        X_fold_validation = train_data.loc[
            fold_validation_mask,
            feature_columns,
        ]

        y_fold_validation = train_data.loc[
            fold_validation_mask,
            target_column,
        ].astype("int8")

        tuning_pipeline = Pipeline(
            steps=[
                (
                    "preprocessor",
                    clone(preprocessor),
                ),
                (
                    "classifier",
                    RandomForestClassifier(
                        n_estimators=500,
                        max_depth=(
                            maximum_depth
                        ),
                        min_samples_leaf=(
                            minimum_leaf_size
                        ),
                        max_features="sqrt",
                        class_weight=None,
                        random_state=(
                            RANDOM_STATE
                        ),
                        n_jobs=-1,
                    ),
                ),
            ]
        )

        tuning_pipeline.fit(
            X_fold_train,
            y_fold_train,
        )

        fold_probability = (
            tuning_pipeline.predict_proba(
                X_fold_validation
            )[:, 1]
        )

        fold_pr_auc = (
            average_precision_score(
                y_fold_validation,
                fold_probability,
            )
        )

        fold_roc_auc = roc_auc_score(
            y_fold_validation,
            fold_probability,
        )

        fold_brier = brier_score_loss(
            y_fold_validation,
            fold_probability,
        )

        rf_tuning_result_rows.append(
            {
                "Configuration": (
                    configuration_name
                ),
                "Maximum depth": (
                    "Unlimited"
                    if maximum_depth is None
                    else maximum_depth
                ),
                "Minimum leaf size": (
                    minimum_leaf_size
                ),
                "Fold": fold_number,
                "Validation year": (
                    validation_year
                ),
                "Validation events": (
                    y_fold_validation.sum()
                ),
                "ROC-AUC": fold_roc_auc,
                "PR-AUC": fold_pr_auc,
                "PR-AUC baseline": (
                    y_fold_validation.mean()
                ),
                "PR-AUC lift": (
                    fold_pr_auc
                    / y_fold_validation.mean()
                ),
                "Brier score": fold_brier,
            }
        )

rf_tuning_results = pd.DataFrame(
    rf_tuning_result_rows
)

# ---------------------------------------------------------
# Summarise each configuration across time
# ---------------------------------------------------------

rf_tuning_summary = (
    rf_tuning_results
    .groupby(
        [
            "Configuration",
            "Maximum depth",
            "Minimum leaf size",
        ],
        observed=True,
        dropna=False,
    )
    .agg(
        Mean_ROC_AUC=(
            "ROC-AUC",
            "mean",
        ),
        Minimum_ROC_AUC=(
            "ROC-AUC",
            "min",
        ),
        Mean_PR_AUC=(
            "PR-AUC",
            "mean",
        ),
        Minimum_PR_AUC=(
            "PR-AUC",
            "min",
        ),
        Maximum_PR_AUC=(
            "PR-AUC",
            "max",
        ),
        PR_AUC_standard_deviation=(
            "PR-AUC",
            "std",
        ),
        Mean_PR_AUC_lift=(
            "PR-AUC lift",
            "mean",
        ),
        Mean_Brier_score=(
            "Brier score",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "Mean_PR_AUC",
            "Minimum_PR_AUC",
            "Mean_Brier_score",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

rf_tuning_summary[
    "Temporal rank"
] = (
    np.arange(
        1,
        len(rf_tuning_summary) + 1,
    )
)

print(
    "Random-forest temporal tuning summary:"
)
display(rf_tuning_summary)

# Show yearly PR-AUC values
rf_tuning_year_pivot = (
    rf_tuning_results
    .pivot(
        index="Configuration",
        columns="Validation year",
        values="PR-AUC",
    )
    .reset_index()
)

print(
    "\nRandom-forest PR-AUC by "
    "configuration and validation year:"
)
display(rf_tuning_year_pivot)

# ---------------------------------------------------------
# Select the strongest temporally stable configuration
# ---------------------------------------------------------

selected_rf_summary = (
    rf_tuning_summary.iloc[0]
)

selected_rf_configuration_name = (
    selected_rf_summary[
        "Configuration"
    ]
)

selected_rf_configuration = next(
    configuration
    for configuration in (
        random_forest_tuning_configurations
    )
    if configuration[
        "Configuration"
    ]
    == selected_rf_configuration_name
)

selected_rf_parameters = pd.Series(
    {
        "Configuration": (
            selected_rf_configuration_name
        ),
        "Trees": 500,
        "Maximum depth": (
            "Unlimited"
            if selected_rf_configuration[
                "Maximum depth"
            ] is None
            else selected_rf_configuration[
                "Maximum depth"
            ]
        ),
        "Minimum observations per leaf": (
            selected_rf_configuration[
                "Minimum leaf size"
            ]
        ),
        "Features considered per split": (
            "Square root of available features"
        ),
        "Class weighting": "None",
        "Selection basis": (
            "Highest mean temporal PR-AUC; "
            "minimum PR-AUC and Brier score "
            "used as secondary checks"
        ),
    },
    name="Selected setting",
).to_frame()

print(
    "\nSelected random-forest configuration:"
)
display(selected_rf_parameters)

# ---------------------------------------------------------
# Refit the selected configuration on all training years
# ---------------------------------------------------------

tuned_rf_model_name = (
    "Temporally tuned random forest"
)

tuned_rf_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor),
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=(
                    selected_rf_configuration[
                        "Maximum depth"
                    ]
                ),
                min_samples_leaf=(
                    selected_rf_configuration[
                        "Minimum leaf size"
                    ]
                ),
                max_features="sqrt",
                class_weight=None,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

tuned_rf_pipeline.fit(
    X_train,
    y_train,
)

tuned_rf_validation_probability = (
    tuned_rf_pipeline.predict_proba(
        X_validation
    )[:, 1]
)

# Find the F2-oriented validation threshold
tuned_rf_threshold_rows = []

for threshold in threshold_grid:
    result, _ = evaluate_probabilities(
        y_true=y_validation,
        predicted_probability=(
            tuned_rf_validation_probability
        ),
        threshold=threshold,
        model_name=tuned_rf_model_name,
        dataset_name="Validation",
    )

    tuned_rf_threshold_rows.append(
        result
    )

tuned_rf_threshold_results = pd.DataFrame(
    tuned_rf_threshold_rows
)

tuned_rf_best_f2_row = (
    tuned_rf_threshold_results
    .sort_values(
        [
            "F2",
            "Recall",
            "Precision",
            "Threshold",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .iloc[0]
)

tuned_rf_selected_threshold = float(
    tuned_rf_best_f2_row[
        "Threshold"
    ]
)

fitted_development_models[
    tuned_rf_model_name
] = tuned_rf_pipeline

all_validation_probabilities[
    tuned_rf_model_name
] = (
    tuned_rf_validation_probability
)

selected_f2_thresholds[
    tuned_rf_model_name
] = (
    tuned_rf_selected_threshold
)

# ---------------------------------------------------------
# Compare the tuned forest with the leading logistic model
# ---------------------------------------------------------

leading_model_comparison_rows = []

for model_name in [
    "Ordinary logistic regression",
    tuned_rf_model_name,
]:
    probabilities = (
        all_validation_probabilities[
            model_name
        ]
    )

    threshold = (
        selected_f2_thresholds[
            model_name
        ]
    )

    result, _ = evaluate_probabilities(
        y_true=y_validation,
        predicted_probability=(
            probabilities
        ),
        threshold=threshold,
        model_name=model_name,
        dataset_name="Validation",
    )

    leading_model_comparison_rows.append(
        {
            "Model": model_name,
            "Selected threshold": (
                threshold
            ),
            "Warnings issued": (
                result["Predicted events"]
            ),
            "Shortages detected": (
                result["True positives"]
            ),
            "Shortages missed": (
                result["False negatives"]
            ),
            "False warnings": (
                result["False positives"]
            ),
            "Precision": (
                result["Precision"]
            ),
            "Recall": result["Recall"],
            "F1": result["F1"],
            "F2": result["F2"],
            "Balanced accuracy": (
                result[
                    "Balanced accuracy"
                ]
            ),
            "ROC-AUC": (
                result["ROC-AUC"]
            ),
            "PR-AUC": (
                result["PR-AUC"]
            ),
            "Brier score": (
                result["Brier score"]
            ),
        }
    )

leading_model_comparison = pd.DataFrame(
    leading_model_comparison_rows
)

print(
    "\nLeading validation-model comparison:"
)
display(leading_model_comparison)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert len(
    rf_tuning_results
) == 24

assert len(
    rf_tuning_summary
) == 6

assert rf_tuning_results[
    "Validation year"
].isin(
    [2015, 2016, 2017, 2018]
).all()

assert (
    selected_rf_configuration_name
    in {
        configuration[
            "Configuration"
        ]
        for configuration in (
            random_forest_tuning_configurations
        )
    }
)

assert len(
    tuned_rf_validation_probability
) == 500

assert np.isfinite(
    tuned_rf_validation_probability
).all()

assert (
    tuned_rf_validation_probability
    >= 0
).all()

assert (
    tuned_rf_validation_probability
    <= 1
).all()

assert (
    0.01
    <= tuned_rf_selected_threshold
    <= 0.99
)

print(
    "\nRandom-forest tuning and validation "
    "completed successfully."
)

print(
    "All complexity settings were selected using "
    "2015–2018 temporal folds. The test period "
    "remains locked."
)

Random-forest temporal tuning summary:


,Configuration,Maximum depth,Minimum leaf size,Mean_ROC_AUC,Minimum_ROC_AUC,Mean_PR_AUC,Minimum_PR_AUC,Maximum_PR_AUC,PR_AUC_standard_deviation,Mean_PR_AUC_lift,Mean_Brier_score,Temporal rank
0,RF_A,Unlimited,3,0.7314,0.6739,0.3317,0.1980,0.4990,0.1248,3.3332,0.0847,1
1,RF_C,8,3,0.7281,0.6812,0.3217,0.1916,0.4672,0.1132,3.2242,0.0846,2
2,RF_F,8,6,0.7284,0.6772,0.3195,0.1595,0.4865,0.1346,3.1390,0.0848,3
3,RF_D,Unlimited,6,0.7263,0.6745,0.3192,0.1595,0.5024,0.1408,3.1393,0.0847,4
4,RF_B,12,3,0.7297,0.6751,0.3191,0.1612,0.4917,0.1354,3.1389,0.0847,5
5,RF_E,12,6,0.7275,0.6799,0.3138,0.1539,0.4990,0.1425,3.0848,0.0847,6



Random-forest PR-AUC by configuration and validation year:


Validation year,Configuration,2015,2016,2017,2018
0,RF_A,0.3259,0.1980,0.4990,0.3039
1,RF_B,0.3204,0.1612,0.4917,0.3029
2,RF_C,0.3234,0.1916,0.4672,0.3044
3,RF_D,0.3145,0.1595,0.5024,0.3002
4,RF_E,0.3204,0.1539,0.4990,0.2817
5,RF_F,0.3360,0.1595,0.4865,0.2960



Selected random-forest configuration:


,Selected setting
Configuration,RF_A
Trees,500
Maximum depth,Unlimited
Minimum observations per leaf,3
Features considered per split,Square root of available features
Class weighting,None
Selection basis,Highest mean temporal PR-AUC; minimum PR-AUC a...



Leading validation-model comparison:


,Model,Selected threshold,Warnings issued,Shortages detected,Shortages missed,False warnings,Precision,Recall,F1,F2,Balanced accuracy,ROC-AUC,PR-AUC,Brier score
0,Ordinary logistic regression,0.1300,157,36,16,121,0.2293,0.6923,0.3445,0.4932,0.7111,0.7172,0.3225,0.0859
1,Temporally tuned random forest,0.1200,189,37,15,152,0.1958,0.7115,0.3071,0.4660,0.6861,0.7395,0.2835,0.0869



Random-forest tuning and validation completed successfully.
All complexity settings were selected using 2015–2018 temporal folds. The test period remains locked.


In [10]:
# ---------------------------------------------------------
# Define the regularisation search
# ---------------------------------------------------------

logistic_tuning_configurations = [
    {
        "Configuration": "LR_A",
        "C": 0.01,
    },
    {
        "Configuration": "LR_B",
        "C": 0.03,
    },
    {
        "Configuration": "LR_C",
        "C": 0.10,
    },
    {
        "Configuration": "LR_D",
        "C": 0.30,
    },
    {
        "Configuration": "LR_E",
        "C": 1.00,
    },
    {
        "Configuration": "LR_F",
        "C": 3.00,
    },
    {
        "Configuration": "LR_G",
        "C": 10.00,
    },
]

logistic_tuning_result_rows = []

# ---------------------------------------------------------
# Evaluate every setting across the temporal folds
# ---------------------------------------------------------

for configuration in (
    logistic_tuning_configurations
):
    configuration_name = (
        configuration["Configuration"]
    )

    regularisation_c = (
        configuration["C"]
    )

    for fold_definition in (
        temporal_cv_definitions
    ):
        fold_number = (
            fold_definition["Fold"]
        )

        train_start_year = (
            fold_definition[
                "Training year start"
            ]
        )

        train_end_year = (
            fold_definition[
                "Training year end"
            ]
        )

        validation_year = (
            fold_definition[
                "Validation year"
            ]
        )

        fold_train_mask = (
            train_data["Year"]
            .between(
                train_start_year,
                train_end_year,
            )
        )

        fold_validation_mask = (
            train_data["Year"]
            .eq(validation_year)
        )

        X_fold_train = train_data.loc[
            fold_train_mask,
            feature_columns,
        ]

        y_fold_train = train_data.loc[
            fold_train_mask,
            target_column,
        ].astype("int8")

        X_fold_validation = train_data.loc[
            fold_validation_mask,
            feature_columns,
        ]

        y_fold_validation = train_data.loc[
            fold_validation_mask,
            target_column,
        ].astype("int8")

        tuning_pipeline = Pipeline(
            steps=[
                (
                    "preprocessor",
                    clone(preprocessor),
                ),
                (
                    "classifier",
                    LogisticRegression(
                        C=regularisation_c,
                        l1_ratio=0,
                        class_weight=None,
                        solver="lbfgs",
                        max_iter=5000,
                        random_state=(
                            RANDOM_STATE
                        ),
                    ),
                ),
            ]
        )

        tuning_pipeline.fit(
            X_fold_train,
            y_fold_train,
        )

        fold_probability = (
            tuning_pipeline.predict_proba(
                X_fold_validation
            )[:, 1]
        )

        fold_pr_auc = (
            average_precision_score(
                y_fold_validation,
                fold_probability,
            )
        )

        fold_roc_auc = roc_auc_score(
            y_fold_validation,
            fold_probability,
        )

        fold_brier = brier_score_loss(
            y_fold_validation,
            fold_probability,
        )

        logistic_tuning_result_rows.append(
            {
                "Configuration": (
                    configuration_name
                ),
                "C": regularisation_c,
                "Fold": fold_number,
                "Validation year": (
                    validation_year
                ),
                "Validation events": (
                    y_fold_validation.sum()
                ),
                "ROC-AUC": fold_roc_auc,
                "PR-AUC": fold_pr_auc,
                "PR-AUC baseline": (
                    y_fold_validation.mean()
                ),
                "PR-AUC lift": (
                    fold_pr_auc
                    / y_fold_validation.mean()
                ),
                "Brier score": fold_brier,
            }
        )

logistic_tuning_results = pd.DataFrame(
    logistic_tuning_result_rows
)

# ---------------------------------------------------------
# Summarise temporal performance
# ---------------------------------------------------------

logistic_tuning_summary = (
    logistic_tuning_results
    .groupby(
        [
            "Configuration",
            "C",
        ],
        observed=True,
    )
    .agg(
        Mean_ROC_AUC=(
            "ROC-AUC",
            "mean",
        ),
        Minimum_ROC_AUC=(
            "ROC-AUC",
            "min",
        ),
        Mean_PR_AUC=(
            "PR-AUC",
            "mean",
        ),
        Minimum_PR_AUC=(
            "PR-AUC",
            "min",
        ),
        Maximum_PR_AUC=(
            "PR-AUC",
            "max",
        ),
        PR_AUC_standard_deviation=(
            "PR-AUC",
            "std",
        ),
        Mean_PR_AUC_lift=(
            "PR-AUC lift",
            "mean",
        ),
        Mean_Brier_score=(
            "Brier score",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "Mean_PR_AUC",
            "Minimum_PR_AUC",
            "Mean_Brier_score",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

logistic_tuning_summary[
    "Temporal rank"
] = np.arange(
    1,
    len(logistic_tuning_summary) + 1,
)

print(
    "Logistic-regression temporal "
    "tuning summary:"
)
display(logistic_tuning_summary)

logistic_tuning_year_pivot = (
    logistic_tuning_results
    .pivot(
        index="Configuration",
        columns="Validation year",
        values="PR-AUC",
    )
    .reset_index()
)

print(
    "\nLogistic-regression PR-AUC by "
    "configuration and validation year:"
)
display(logistic_tuning_year_pivot)

# ---------------------------------------------------------
# Select the strongest temporal configuration
# ---------------------------------------------------------

selected_logistic_summary = (
    logistic_tuning_summary.iloc[0]
)

selected_logistic_configuration_name = (
    selected_logistic_summary[
        "Configuration"
    ]
)

selected_logistic_configuration = next(
    configuration
    for configuration in (
        logistic_tuning_configurations
    )
    if configuration[
        "Configuration"
    ]
    == selected_logistic_configuration_name
)

selected_logistic_c = float(
    selected_logistic_configuration["C"]
)

selected_logistic_parameters = pd.Series(
    {
        "Configuration": (
            selected_logistic_configuration_name
        ),
        "C": selected_logistic_c,
        "Regularisation strength": (
            "Stronger"
            if selected_logistic_c < 1
            else (
                "Standard"
                if selected_logistic_c == 1
                else "Weaker"
            )
        ),
        "Class weighting": "None",
        "Selection basis": (
            "Highest mean temporal PR-AUC; "
            "minimum PR-AUC and Brier score "
            "used as secondary checks"
        ),
    },
    name="Selected setting",
).to_frame()

print(
    "\nSelected logistic-regression configuration:"
)
display(selected_logistic_parameters)

# ---------------------------------------------------------
# Refit the selected setting on all training years
# ---------------------------------------------------------

tuned_logistic_model_name = (
    "Temporally tuned logistic regression"
)

tuned_logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor),
        ),
        (
            "classifier",
            LogisticRegression(
                C=selected_logistic_c,
                l1_ratio=0,
                class_weight=None,
                solver="lbfgs",
                max_iter=5000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

tuned_logistic_pipeline.fit(
    X_train,
    y_train,
)

tuned_logistic_validation_probability = (
    tuned_logistic_pipeline.predict_proba(
        X_validation
    )[:, 1]
)

# Select an F2-oriented threshold on validation data
tuned_logistic_threshold_rows = []

for threshold in threshold_grid:
    result, _ = evaluate_probabilities(
        y_true=y_validation,
        predicted_probability=(
            tuned_logistic_validation_probability
        ),
        threshold=threshold,
        model_name=tuned_logistic_model_name,
        dataset_name="Validation",
    )

    tuned_logistic_threshold_rows.append(
        result
    )

tuned_logistic_threshold_results = (
    pd.DataFrame(
        tuned_logistic_threshold_rows
    )
)

tuned_logistic_best_f2_row = (
    tuned_logistic_threshold_results
    .sort_values(
        [
            "F2",
            "Recall",
            "Precision",
            "Threshold",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .iloc[0]
)

tuned_logistic_selected_threshold = float(
    tuned_logistic_best_f2_row[
        "Threshold"
    ]
)

fitted_development_models[
    tuned_logistic_model_name
] = (
    tuned_logistic_pipeline
)

all_validation_probabilities[
    tuned_logistic_model_name
] = (
    tuned_logistic_validation_probability
)

selected_f2_thresholds[
    tuned_logistic_model_name
] = (
    tuned_logistic_selected_threshold
)

# ---------------------------------------------------------
# Compare the two temporally tuned finalists
# ---------------------------------------------------------

finalist_comparison_rows = []

for model_name in [
    tuned_logistic_model_name,
    tuned_rf_model_name,
]:
    probabilities = (
        all_validation_probabilities[
            model_name
        ]
    )

    threshold = (
        selected_f2_thresholds[
            model_name
        ]
    )

    result, _ = evaluate_probabilities(
        y_true=y_validation,
        predicted_probability=(
            probabilities
        ),
        threshold=threshold,
        model_name=model_name,
        dataset_name="Validation",
    )

    finalist_comparison_rows.append(
        {
            "Model": model_name,
            "Selected threshold": (
                threshold
            ),
            "Warnings issued": (
                result["Predicted events"]
            ),
            "Shortages detected": (
                result["True positives"]
            ),
            "Shortages missed": (
                result["False negatives"]
            ),
            "False warnings": (
                result["False positives"]
            ),
            "Precision": (
                result["Precision"]
            ),
            "Recall": result["Recall"],
            "F1": result["F1"],
            "F2": result["F2"],
            "Balanced accuracy": (
                result[
                    "Balanced accuracy"
                ]
            ),
            "ROC-AUC": (
                result["ROC-AUC"]
            ),
            "PR-AUC": (
                result["PR-AUC"]
            ),
            "Brier score": (
                result["Brier score"]
            ),
        }
    )

finalist_validation_comparison = (
    pd.DataFrame(
        finalist_comparison_rows
    )
)

print(
    "\nTemporally tuned finalist "
    "validation comparison:"
)
display(finalist_validation_comparison)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert len(
    logistic_tuning_results
) == 28

assert len(
    logistic_tuning_summary
) == 7

assert logistic_tuning_results[
    "Validation year"
].isin(
    [2015, 2016, 2017, 2018]
).all()

assert (
    selected_logistic_configuration_name
    in {
        configuration[
            "Configuration"
        ]
        for configuration in (
            logistic_tuning_configurations
        )
    }
)

assert len(
    tuned_logistic_validation_probability
) == 500

assert np.isfinite(
    tuned_logistic_validation_probability
).all()

assert (
    tuned_logistic_validation_probability
    >= 0
).all()

assert (
    tuned_logistic_validation_probability
    <= 1
).all()

assert (
    0.01
    <= tuned_logistic_selected_threshold
    <= 0.99
)

print(
    "\nLogistic-regression regularisation tuning "
    "completed successfully."
)

print(
    "The locked test period remains unused."
)

Logistic-regression temporal tuning summary:


,Configuration,C,Mean_ROC_AUC,Minimum_ROC_AUC,Mean_PR_AUC,Minimum_PR_AUC,Maximum_PR_AUC,PR_AUC_standard_deviation,Mean_PR_AUC_lift,Mean_Brier_score,Temporal rank
0,LR_A,0.0100,0.6976,0.6341,0.2612,0.1289,0.3542,0.0961,2.5331,0.0874,1
1,LR_B,0.0300,0.6954,0.6295,0.2511,0.1420,0.3245,0.0782,2.4743,0.0886,2
2,LR_C,0.1000,0.6923,0.6204,0.2434,0.1428,0.3035,0.0722,2.4152,0.0901,3
3,LR_D,0.3000,0.6932,0.6152,0.2425,0.1373,0.3024,0.0777,2.4016,0.0916,4
4,LR_E,1.0000,0.6933,0.6145,0.2389,0.1395,0.3004,0.0743,2.3770,0.0943,5
5,LR_F,3.0000,0.6890,0.6065,0.2359,0.1353,0.3046,0.0760,2.3440,0.0973,6
6,LR_G,10.0000,0.6894,0.6078,0.2344,0.1448,0.3035,0.0687,2.3528,0.1001,7



Logistic-regression PR-AUC by configuration and validation year:


Validation year,Configuration,2015,2016,2017,2018
0,LR_A,0.3542,0.1289,0.3003,0.2616
1,LR_B,0.3245,0.1420,0.2836,0.2541
2,LR_C,0.3035,0.1428,0.2867,0.2406
3,LR_D,0.3024,0.1373,0.3001,0.2304
4,LR_E,0.2908,0.1395,0.3004,0.2250
5,LR_F,0.2832,0.1353,0.3046,0.2203
6,LR_G,0.2687,0.1448,0.3035,0.2205



Selected logistic-regression configuration:


,Selected setting
Configuration,LR_A
C,0.0100
Regularisation strength,Stronger
Class weighting,None
Selection basis,Highest mean temporal PR-AUC; minimum PR-AUC a...



Temporally tuned finalist validation comparison:


,Model,Selected threshold,Warnings issued,Shortages detected,Shortages missed,False warnings,Precision,Recall,F1,F2,Balanced accuracy,ROC-AUC,PR-AUC,Brier score
0,Temporally tuned logistic regression,0.1500,118,33,19,85,0.2797,0.6346,0.3882,0.5061,0.7224,0.7216,0.3042,0.0858
1,Temporally tuned random forest,0.1200,189,37,15,152,0.1958,0.7115,0.3071,0.4660,0.6861,0.7395,0.2835,0.0869



Logistic-regression regularisation tuning completed successfully.
The locked test period remains unused.


In [11]:
# ---------------------------------------------------------
# Define candidate ensemble weights
# ---------------------------------------------------------

# Weight refers to logistic regression.
# Random-forest weight is 1 minus this value.
ensemble_weight_grid = [
    0.00,
    0.25,
    0.50,
    0.75,
    1.00,
]

ensemble_temporal_prediction_store = {}

# ---------------------------------------------------------
# Generate finalist probabilities for each temporal fold
# ---------------------------------------------------------

for fold_definition in temporal_cv_definitions:
    fold_number = (
        fold_definition["Fold"]
    )

    train_start_year = (
        fold_definition[
            "Training year start"
        ]
    )

    train_end_year = (
        fold_definition[
            "Training year end"
        ]
    )

    validation_year = (
        fold_definition[
            "Validation year"
        ]
    )

    fold_train_mask = (
        train_data["Year"]
        .between(
            train_start_year,
            train_end_year,
        )
    )

    fold_validation_mask = (
        train_data["Year"]
        .eq(validation_year)
    )

    X_fold_train = train_data.loc[
        fold_train_mask,
        feature_columns,
    ]

    y_fold_train = train_data.loc[
        fold_train_mask,
        target_column,
    ].astype("int8")

    X_fold_validation = train_data.loc[
        fold_validation_mask,
        feature_columns,
    ]

    y_fold_validation = train_data.loc[
        fold_validation_mask,
        target_column,
    ].astype("int8")

    fold_logistic_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                clone(preprocessor),
            ),
            (
                "classifier",
                LogisticRegression(
                    C=selected_logistic_c,
                    l1_ratio=0,
                    class_weight=None,
                    solver="lbfgs",
                    max_iter=5000,
                    random_state=(
                        RANDOM_STATE
                    ),
                ),
            ),
        ]
    )

    fold_forest_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                clone(preprocessor),
            ),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=(
                        selected_rf_configuration[
                            "Maximum depth"
                        ]
                    ),
                    min_samples_leaf=(
                        selected_rf_configuration[
                            "Minimum leaf size"
                        ]
                    ),
                    max_features="sqrt",
                    class_weight=None,
                    random_state=(
                        RANDOM_STATE
                    ),
                    n_jobs=-1,
                ),
            ),
        ]
    )

    fold_logistic_pipeline.fit(
        X_fold_train,
        y_fold_train,
    )

    fold_forest_pipeline.fit(
        X_fold_train,
        y_fold_train,
    )

    logistic_probability = (
        fold_logistic_pipeline
        .predict_proba(
            X_fold_validation
        )[:, 1]
    )

    forest_probability = (
        fold_forest_pipeline
        .predict_proba(
            X_fold_validation
        )[:, 1]
    )

    ensemble_temporal_prediction_store[
        fold_number
    ] = {
        "Validation year": (
            validation_year
        ),
        "y_true": (
            y_fold_validation
            .to_numpy()
        ),
        "Logistic probability": (
            logistic_probability
        ),
        "Forest probability": (
            forest_probability
        ),
    }

# ---------------------------------------------------------
# Evaluate each weight across the earlier years
# ---------------------------------------------------------

ensemble_temporal_result_rows = []

for logistic_weight in (
    ensemble_weight_grid
):
    forest_weight = (
        1.0 - logistic_weight
    )

    for fold_number, fold_values in (
        ensemble_temporal_prediction_store
        .items()
    ):
        y_fold_true = (
            fold_values["y_true"]
        )

        ensemble_probability = (
            logistic_weight
            * fold_values[
                "Logistic probability"
            ]
            + forest_weight
            * fold_values[
                "Forest probability"
            ]
        )

        fold_event_rate = (
            y_fold_true.mean()
        )

        ensemble_temporal_result_rows.append(
            {
                "Logistic weight": (
                    logistic_weight
                ),
                "Forest weight": (
                    forest_weight
                ),
                "Fold": fold_number,
                "Validation year": (
                    fold_values[
                        "Validation year"
                    ]
                ),
                "ROC-AUC": roc_auc_score(
                    y_fold_true,
                    ensemble_probability,
                ),
                "PR-AUC": (
                    average_precision_score(
                        y_fold_true,
                        ensemble_probability,
                    )
                ),
                "PR-AUC baseline": (
                    fold_event_rate
                ),
                "Brier score": (
                    brier_score_loss(
                        y_fold_true,
                        ensemble_probability,
                    )
                ),
            }
        )

ensemble_temporal_results = pd.DataFrame(
    ensemble_temporal_result_rows
)

ensemble_temporal_results[
    "PR-AUC lift"
] = (
    ensemble_temporal_results[
        "PR-AUC"
    ]
    / ensemble_temporal_results[
        "PR-AUC baseline"
    ]
)

ensemble_weight_summary = (
    ensemble_temporal_results
    .groupby(
        [
            "Logistic weight",
            "Forest weight",
        ],
        observed=True,
    )
    .agg(
        Mean_ROC_AUC=(
            "ROC-AUC",
            "mean",
        ),
        Minimum_ROC_AUC=(
            "ROC-AUC",
            "min",
        ),
        Mean_PR_AUC=(
            "PR-AUC",
            "mean",
        ),
        Minimum_PR_AUC=(
            "PR-AUC",
            "min",
        ),
        Maximum_PR_AUC=(
            "PR-AUC",
            "max",
        ),
        PR_AUC_standard_deviation=(
            "PR-AUC",
            "std",
        ),
        Mean_PR_AUC_lift=(
            "PR-AUC lift",
            "mean",
        ),
        Mean_Brier_score=(
            "Brier score",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "Mean_PR_AUC",
            "Minimum_PR_AUC",
            "Mean_Brier_score",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

ensemble_weight_summary[
    "Temporal rank"
] = np.arange(
    1,
    len(ensemble_weight_summary) + 1,
)

print(
    "Temporal ensemble-weight comparison:"
)
display(ensemble_weight_summary)

# Show yearly PR-AUC
ensemble_yearly_pr_auc = (
    ensemble_temporal_results
    .assign(
        Weight_label=lambda frame: (
            "Logistic "
            + frame[
                "Logistic weight"
            ].map(
                lambda value: (
                    f"{value:.2f}"
                )
            )
            + " / Forest "
            + frame[
                "Forest weight"
            ].map(
                lambda value: (
                    f"{value:.2f}"
                )
            )
        )
    )
    .pivot(
        index="Weight_label",
        columns="Validation year",
        values="PR-AUC",
    )
    .reset_index()
)

print(
    "\nEnsemble PR-AUC by validation year:"
)
display(ensemble_yearly_pr_auc)

# ---------------------------------------------------------
# Select the strongest earlier-year weight
# ---------------------------------------------------------

selected_ensemble_summary = (
    ensemble_weight_summary.iloc[0]
)

selected_logistic_weight = float(
    selected_ensemble_summary[
        "Logistic weight"
    ]
)

selected_forest_weight = float(
    selected_ensemble_summary[
        "Forest weight"
    ]
)

selected_ensemble_parameters = pd.Series(
    {
        "Logistic-regression weight": (
            selected_logistic_weight
        ),
        "Random-forest weight": (
            selected_forest_weight
        ),
        "Selection basis": (
            "Highest mean 2015–2018 PR-AUC; "
            "minimum PR-AUC and Brier score "
            "used as secondary checks"
        ),
    },
    name="Selected setting",
).to_frame()

print("\nSelected ensemble weighting:")
display(selected_ensemble_parameters)

# ---------------------------------------------------------
# Apply the selected weight to main validation probabilities
# ---------------------------------------------------------

ensemble_model_name = (
    "Temporally selected probability ensemble"
)

ensemble_validation_probability = (
    selected_logistic_weight
    * tuned_logistic_validation_probability
    + selected_forest_weight
    * tuned_rf_validation_probability
)

ensemble_validation_threshold_rows = []

for threshold in threshold_grid:
    result, _ = evaluate_probabilities(
        y_true=y_validation,
        predicted_probability=(
            ensemble_validation_probability
        ),
        threshold=threshold,
        model_name=ensemble_model_name,
        dataset_name="Validation",
    )

    ensemble_validation_threshold_rows.append(
        result
    )

ensemble_validation_threshold_results = (
    pd.DataFrame(
        ensemble_validation_threshold_rows
    )
)

ensemble_best_f2_row = (
    ensemble_validation_threshold_results
    .sort_values(
        [
            "F2",
            "Recall",
            "Precision",
            "Threshold",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .iloc[0]
)

ensemble_selected_threshold = float(
    ensemble_best_f2_row[
        "Threshold"
    ]
)

selected_f2_thresholds[
    ensemble_model_name
] = ensemble_selected_threshold

all_validation_probabilities[
    ensemble_model_name
] = ensemble_validation_probability

# ---------------------------------------------------------
# Compare all three finalists
# ---------------------------------------------------------

three_finalist_rows = []

for model_name, probabilities in [
    (
        tuned_logistic_model_name,
        tuned_logistic_validation_probability,
    ),
    (
        tuned_rf_model_name,
        tuned_rf_validation_probability,
    ),
    (
        ensemble_model_name,
        ensemble_validation_probability,
    ),
]:
    threshold = (
        selected_f2_thresholds[
            model_name
        ]
    )

    result, _ = evaluate_probabilities(
        y_true=y_validation,
        predicted_probability=(
            probabilities
        ),
        threshold=threshold,
        model_name=model_name,
        dataset_name="Validation",
    )

    three_finalist_rows.append(
        {
            "Model": model_name,
            "Selected threshold": (
                threshold
            ),
            "Warnings issued": (
                result["Predicted events"]
            ),
            "Shortages detected": (
                result["True positives"]
            ),
            "Shortages missed": (
                result["False negatives"]
            ),
            "False warnings": (
                result["False positives"]
            ),
            "Precision": (
                result["Precision"]
            ),
            "Recall": result["Recall"],
            "F1": result["F1"],
            "F2": result["F2"],
            "Balanced accuracy": (
                result[
                    "Balanced accuracy"
                ]
            ),
            "ROC-AUC": (
                result["ROC-AUC"]
            ),
            "PR-AUC": (
                result["PR-AUC"]
            ),
            "Brier score": (
                result["Brier score"]
            ),
        }
    )

three_finalist_comparison = (
    pd.DataFrame(
        three_finalist_rows
    )
)

print(
    "\nThree-finalist validation comparison:"
)
display(three_finalist_comparison)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert len(
    ensemble_temporal_prediction_store
) == 4

assert len(
    ensemble_temporal_results
) == 20

assert len(
    ensemble_weight_summary
) == 5

assert np.isclose(
    selected_logistic_weight
    + selected_forest_weight,
    1.0,
)

assert len(
    ensemble_validation_probability
) == 500

assert np.isfinite(
    ensemble_validation_probability
).all()

assert (
    ensemble_validation_probability
    >= 0
).all()

assert (
    ensemble_validation_probability
    <= 1
).all()

assert (
    0.01
    <= ensemble_selected_threshold
    <= 0.99
)

print(
    "\nTemporal ensemble testing completed "
    "successfully."
)

print(
    "The selected weighting used 2015–2018 only. "
    "The warning threshold used 2019–2020 validation "
    "only. The test period remains locked."
)

Temporal ensemble-weight comparison:


,Logistic weight,Forest weight,Mean_ROC_AUC,Minimum_ROC_AUC,Mean_PR_AUC,Minimum_PR_AUC,Maximum_PR_AUC,PR_AUC_standard_deviation,Mean_PR_AUC_lift,Mean_Brier_score,Temporal rank
0,0.0000,1.0000,0.7314,0.6739,0.3317,0.1980,0.4990,0.1248,3.3332,0.0847,1
1,0.2500,0.7500,0.7406,0.6957,0.3230,0.1646,0.4793,0.1297,3.1770,0.0844,2
2,0.5000,0.5000,0.7411,0.6993,0.3073,0.1474,0.4419,0.1232,2.9968,0.0848,3
3,0.7500,0.2500,0.7317,0.6821,0.2889,0.1420,0.3733,0.1064,2.8132,0.0858,4
4,1.0000,0.0000,0.6976,0.6341,0.2612,0.1289,0.3542,0.0961,2.5331,0.0874,5



Ensemble PR-AUC by validation year:


Validation year,Weight_label,2015,2016,2017,2018
0,Logistic 0.00 / Forest 1.00,0.3259,0.1980,0.4990,0.3039
1,Logistic 0.25 / Forest 0.75,0.3451,0.1646,0.4793,0.3030
2,Logistic 0.50 / Forest 0.50,0.3480,0.1474,0.4419,0.2918
3,Logistic 0.75 / Forest 0.25,0.3609,0.1420,0.3733,0.2792
4,Logistic 1.00 / Forest 0.00,0.3542,0.1289,0.3003,0.2616



Selected ensemble weighting:


,Selected setting
Logistic-regression weight,0.0000
Random-forest weight,1.0000
Selection basis,Highest mean 2015–2018 PR-AUC; minimum PR-AUC ...



Three-finalist validation comparison:


,Model,Selected threshold,Warnings issued,Shortages detected,Shortages missed,False warnings,Precision,Recall,F1,F2,Balanced accuracy,ROC-AUC,PR-AUC,Brier score
0,Temporally tuned logistic regression,0.1500,118,33,19,85,0.2797,0.6346,0.3882,0.5061,0.7224,0.7216,0.3042,0.0858
1,Temporally tuned random forest,0.1200,189,37,15,152,0.1958,0.7115,0.3071,0.4660,0.6861,0.7395,0.2835,0.0869
2,Temporally selected probability ensemble,0.1200,189,37,15,152,0.1958,0.7115,0.3071,0.4660,0.6861,0.7395,0.2835,0.0869



Temporal ensemble testing completed successfully.
The selected weighting used 2015–2018 only. The warning threshold used 2019–2020 validation only. The test period remains locked.


In [12]:
from sklearn.inspection import (
    permutation_importance,
)

# ---------------------------------------------------------
# Formally declare the selected model and threshold
# ---------------------------------------------------------

selected_model_name = (
    tuned_rf_model_name
)

selected_model = (
    tuned_rf_pipeline
)

selected_warning_threshold = (
    tuned_rf_selected_threshold
)

selected_model_decision = pd.Series(
    {
        "Selected primary model": (
            selected_model_name
        ),
        "Selected warning threshold": (
            selected_warning_threshold
        ),
        "Model-selection evidence": (
            "Highest mean temporal PR-AUC across "
            "2015–2018"
        ),
        "Validation threshold objective": (
            "Highest validation F2 score"
        ),
        "Validation shortages detected": (
            int(
                tuned_rf_best_f2_row[
                    "True positives"
                ]
            )
        ),
        "Validation shortages missed": (
            int(
                tuned_rf_best_f2_row[
                    "False negatives"
                ]
            )
        ),
        "Validation false warnings": (
            int(
                tuned_rf_best_f2_row[
                    "False positives"
                ]
            )
        ),
        "Validation recall %": (
            100
            * tuned_rf_best_f2_row[
                "Recall"
            ]
        ),
        "Validation precision %": (
            100
            * tuned_rf_best_f2_row[
                "Precision"
            ]
        ),
        "Interpretability benchmark": (
            tuned_logistic_model_name
        ),
        "Test-set switching allowed": False,
    },
    name="Decision",
).to_frame()

print("Development-stage model decision:")
display(selected_model_decision)

# ---------------------------------------------------------
# Create the validation diagnostic table
# ---------------------------------------------------------

selected_validation_probability = (
    tuned_rf_validation_probability
)

selected_validation_prediction = (
    selected_validation_probability
    >= selected_warning_threshold
).astype("int8")

validation_diagnostic = (
    validation_data[
        [
            "observation_id",
            "Area",
            "Item Code",
            "Item",
            "Year",
            "target_year",
            target_column,
        ]
    ]
    .copy()
)

validation_diagnostic[
    "predicted_shortage_probability"
] = selected_validation_probability

validation_diagnostic[
    "predicted_shortage_warning"
] = selected_validation_prediction

# ---------------------------------------------------------
# Reusable subgroup evaluation
# ---------------------------------------------------------

def evaluate_subgroups(
    diagnostic_data,
    grouping_column,
):
    subgroup_rows = []

    for group_value, group in (
        diagnostic_data
        .groupby(
            grouping_column,
            observed=True,
        )
    ):
        y_true_group = (
            group[target_column]
            .to_numpy(dtype=int)
        )

        y_probability_group = (
            group[
                "predicted_shortage_probability"
            ]
            .to_numpy(dtype=float)
        )

        y_prediction_group = (
            group[
                "predicted_shortage_warning"
            ]
            .to_numpy(dtype=int)
        )

        tn, fp, fn, tp = confusion_matrix(
            y_true_group,
            y_prediction_group,
            labels=[0, 1],
        ).ravel()

        if np.unique(
            y_true_group
        ).size == 2:
            subgroup_roc_auc = (
                roc_auc_score(
                    y_true_group,
                    y_probability_group,
                )
            )

            subgroup_pr_auc = (
                average_precision_score(
                    y_true_group,
                    y_probability_group,
                )
            )
        else:
            subgroup_roc_auc = np.nan
            subgroup_pr_auc = np.nan

        subgroup_rows.append(
            {
                grouping_column: group_value,
                "Rows": len(group),
                "Actual shortages": (
                    int(y_true_group.sum())
                ),
                "Warnings issued": (
                    int(y_prediction_group.sum())
                ),
                "Shortages detected": int(tp),
                "Shortages missed": int(fn),
                "False warnings": int(fp),
                "Precision": precision_score(
                    y_true_group,
                    y_prediction_group,
                    zero_division=0,
                ),
                "Recall": recall_score(
                    y_true_group,
                    y_prediction_group,
                    zero_division=0,
                ),
                "F2": fbeta_score(
                    y_true_group,
                    y_prediction_group,
                    beta=2,
                    zero_division=0,
                ),
                "ROC-AUC": subgroup_roc_auc,
                "PR-AUC": subgroup_pr_auc,
            }
        )

    return pd.DataFrame(
        subgroup_rows
    )



Development-stage model decision:


,Decision
Selected primary model,Temporally tuned random forest
Selected warning threshold,0.1200
Model-selection evidence,Highest mean temporal PR-AUC across 2015–2018
Validation threshold objective,Highest validation F2 score
Validation shortages detected,37
Validation shortages missed,15
Validation false warnings,152
Validation recall %,71.1538
Validation precision %,19.5767
Interpretability benchmark,Temporally tuned logistic regression


In [13]:
from sklearn.inspection import (
    permutation_importance,
)

# ---------------------------------------------------------
# Declare the selected development model
# ---------------------------------------------------------

selected_model_name = (
    tuned_rf_model_name
)

selected_model = (
    tuned_rf_pipeline
)

selected_probability_threshold = (
    tuned_rf_selected_threshold
)

selected_validation_probability = (
    tuned_rf_validation_probability
)

selected_validation_result, (
    selected_validation_prediction
) = evaluate_probabilities(
    y_true=y_validation,
    predicted_probability=(
        selected_validation_probability
    ),
    threshold=(
        selected_probability_threshold
    ),
    model_name=selected_model_name,
    dataset_name="Validation",
)

model_selection_record = pd.DataFrame(
    {
        "Decision component": [
            "Primary model",
            "Primary selection evidence",
            "Validation threshold",
            "Threshold objective",
            "Secondary benchmark",
            "Ensemble decision",
            "Test-data status",
        ],
        "Decision": [
            selected_model_name,
            (
                "Highest mean PR-AUC across the "
                "2015–2018 expanding temporal folds"
            ),
            selected_probability_threshold,
            (
                "Highest validation F2, giving recall "
                "more importance than precision"
            ),
            tuned_logistic_model_name,
            (
                "Rejected because temporal selection "
                "assigned 100% weight to random forest"
            ),
            (
                "Still locked and not used for model "
                "or threshold selection"
            ),
        ],
    }
)

print("Provisional model-selection record:")
display(model_selection_record)

# ---------------------------------------------------------
# Build a validation diagnostic table
# ---------------------------------------------------------

validation_diagnostic = (
    validation_data[
        [
            "observation_id",
            "Area",
            "Item Code",
            "Item",
            "Year",
            "target_year",
            target_column,
        ]
    ]
    .copy()
)

validation_diagnostic[
    "predicted_shortage_probability"
] = selected_validation_probability

validation_diagnostic[
    "predicted_shortage_warning"
] = selected_validation_prediction

validation_diagnostic[
    "prediction_status"
] = np.select(
    [
        validation_diagnostic[
            target_column
        ].eq(1)
        & validation_diagnostic[
            "predicted_shortage_warning"
        ].eq(1),

        validation_diagnostic[
            target_column
        ].eq(0)
        & validation_diagnostic[
            "predicted_shortage_warning"
        ].eq(1),

        validation_diagnostic[
            target_column
        ].eq(1)
        & validation_diagnostic[
            "predicted_shortage_warning"
        ].eq(0),
    ],
    [
        "Shortage detected",
        "False warning",
        "Shortage missed",
    ],
    default="Correct non-shortage",
)

# ---------------------------------------------------------
# Reusable subgroup evaluation
# ---------------------------------------------------------

def evaluate_subgroups(
    diagnostic_data,
    grouping_column,
):
    subgroup_rows = []

    for group_value, group_data in (
        diagnostic_data
        .groupby(
            grouping_column,
            observed=True,
        )
    ):
        y_group = (
            group_data[
                target_column
            ].to_numpy(dtype=int)
        )

        probability_group = (
            group_data[
                "predicted_shortage_probability"
            ].to_numpy(dtype=float)
        )

        prediction_group = (
            group_data[
                "predicted_shortage_warning"
            ].to_numpy(dtype=int)
        )

        tn, fp, fn, tp = confusion_matrix(
            y_group,
            prediction_group,
            labels=[0, 1],
        ).ravel()

        if len(np.unique(y_group)) == 2:
            group_roc_auc = roc_auc_score(
                y_group,
                probability_group,
            )

            group_pr_auc = (
                average_precision_score(
                    y_group,
                    probability_group,
                )
            )
        else:
            group_roc_auc = np.nan
            group_pr_auc = np.nan

        subgroup_rows.append(
            {
                grouping_column: group_value,
                "Observations": len(group_data),
                "Actual shortages": (
                    int(y_group.sum())
                ),
                "Warnings issued": (
                    int(
                        prediction_group.sum()
                    )
                ),
                "Shortages detected": int(tp),
                "Shortages missed": int(fn),
                "False warnings": int(fp),
                "Precision": precision_score(
                    y_group,
                    prediction_group,
                    zero_division=0,
                ),
                "Recall": recall_score(
                    y_group,
                    prediction_group,
                    zero_division=0,
                ),
                "F2": fbeta_score(
                    y_group,
                    prediction_group,
                    beta=2,
                    zero_division=0,
                ),
                "ROC-AUC": group_roc_auc,
                "PR-AUC": group_pr_auc,
            }
        )

    return pd.DataFrame(
        subgroup_rows
    )


validation_by_commodity = (
    evaluate_subgroups(
        validation_diagnostic,
        "Item",
    )
    .sort_values(
        [
            "Actual shortages",
            "F2",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

validation_by_year = (
    evaluate_subgroups(
        validation_diagnostic,
        "Year",
    )
    .sort_values("Year")
    .reset_index(drop=True)
)

print(
    "\nSelected-model validation performance "
    "by commodity:"
)
display(validation_by_commodity)

print(
    "\nSelected-model validation performance "
    "by predictor year:"
)
display(validation_by_year)

# ---------------------------------------------------------
# Inspect the highest-risk validation observations
# ---------------------------------------------------------

highest_risk_validation_rows = (
    validation_diagnostic
    .sort_values(
        "predicted_shortage_probability",
        ascending=False,
    )
    .head(20)
    .reset_index(drop=True)
)

print(
    "\nTwenty highest predicted validation risks:"
)

display(
    highest_risk_validation_rows[
        [
            "Area",
            "Item",
            "Year",
            "target_year",
            "predicted_shortage_probability",
            "predicted_shortage_warning",
            target_column,
            "prediction_status",
        ]
    ]
)

# ---------------------------------------------------------
# Calculate original-feature permutation importance
# ---------------------------------------------------------

permutation_result = (
    permutation_importance(
        selected_model,
        X_validation,
        y_validation,
        scoring="average_precision",
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
)

permutation_importance_table = (
    pd.DataFrame(
        {
            "Feature": feature_columns,
            "Mean PR-AUC decrease when shuffled": (
                permutation_result[
                    "importances_mean"
                ]
            ),
            "Importance standard deviation": (
                permutation_result[
                    "importances_std"
                ]
            ),
        }
    )
    .sort_values(
        "Mean PR-AUC decrease when shuffled",
        ascending=False,
    )
    .reset_index(drop=True)
)

permutation_importance_table[
    "Interpretation"
] = np.where(
    permutation_importance_table[
        "Mean PR-AUC decrease when shuffled"
    ].gt(0),
    (
        "Validation ranking worsened when this "
        "feature was disrupted"
    ),
    (
        "No reliable validation improvement "
        "from this feature in this test"
    ),
)

print(
    "\nTwenty most influential original predictors "
    "for validation PR-AUC:"
)

display(
    permutation_importance_table.head(20)
)

# ---------------------------------------------------------
# Summarise the audit
# ---------------------------------------------------------

selection_audit_summary = pd.Series(
    {
        "Selected model": (
            selected_model_name
        ),
        "Selected threshold": (
            selected_probability_threshold
        ),
        "Validation shortages": (
            y_validation.sum()
        ),
        "Validation shortages detected": (
            selected_validation_result[
                "True positives"
            ]
        ),
        "Validation shortages missed": (
            selected_validation_result[
                "False negatives"
            ]
        ),
        "Validation false warnings": (
            selected_validation_result[
                "False positives"
            ]
        ),
        "Validation recall %": (
            100
            * selected_validation_result[
                "Recall"
            ]
        ),
        "Validation precision %": (
            100
            * selected_validation_result[
                "Precision"
            ]
        ),
        "Validation F2": (
            selected_validation_result[
                "F2"
            ]
        ),
        "Commodities evaluated": (
            validation_diagnostic[
                "Item"
            ].nunique()
        ),
        "Validation years evaluated": (
            validation_diagnostic[
                "Year"
            ].nunique()
        ),
    },
    name="Result",
).to_frame()

print("\nSelected-model audit summary:")
display(selection_audit_summary)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert selected_model_name == (
    "Temporally tuned random forest"
)

assert np.isclose(
    selected_probability_threshold,
    0.12,
)

assert len(
    validation_diagnostic
) == 500

assert validation_diagnostic[
    "observation_id"
].is_unique

assert validation_diagnostic[
    "predicted_shortage_probability"
].between(
    0,
    1,
).all()

assert (
    validation_diagnostic[
        "predicted_shortage_warning"
    ].sum()
    == 189
)

assert (
    validation_diagnostic[
        "prediction_status"
    ].value_counts().sum()
    == 500
)

assert len(
    validation_by_commodity
) == 8

assert len(
    validation_by_year
) == 2

assert len(
    permutation_importance_table
) == 129

print(
    "\nThe selected model passed the pre-test "
    "interpretability and subgroup audit."
)

print(
    "The locked test period remains unused."
)

Provisional model-selection record:


,Decision component,Decision
0,Primary model,Temporally tuned random forest
1,Primary selection evidence,Highest mean PR-AUC across the 2015–2018 expan...
2,Validation threshold,0.1200
3,Threshold objective,"Highest validation F2, giving recall more impo..."
4,Secondary benchmark,Temporally tuned logistic regression
5,Ensemble decision,Rejected because temporal selection assigned 1...
6,Test-data status,Still locked and not used for model or thresho...



Selected-model validation performance by commodity:


,Item,Observations,Actual shortages,Warnings issued,Shortages detected,Shortages missed,False warnings,Precision,Recall,F2,ROC-AUC,PR-AUC
0,Wheat and products,86,10,23,7,3,16,0.3043,0.7000,0.5556,0.7855,0.2897
1,Maize and products,82,9,23,7,2,16,0.3043,0.7778,0.5932,0.8600,0.4732
2,Groundnuts,72,9,44,7,2,37,0.1591,0.7778,0.4375,0.7090,0.3628
3,Sorghum and products,52,8,34,7,1,27,0.2059,0.8750,0.5303,0.7443,0.4957
4,Millet and products,42,8,20,5,3,15,0.2500,0.6250,0.4808,0.6838,0.4340
5,Cassava and products,55,4,10,2,2,8,0.2000,0.5000,0.3846,0.7402,0.2633
6,Rice and products,86,3,32,2,1,30,0.0625,0.6667,0.2273,0.5904,0.0591
7,Yams,25,1,3,0,1,3,0.0000,0.0000,0.0000,0.3333,0.0588



Selected-model validation performance by predictor year:


...


Twenty most influential original predictors for validation PR-AUC:


,Feature,Mean PR-AUC decrease when shuffled,Importance standard deviation,Interpretation
0,food_1000t_pct_change1,0.0097,0.0044,Validation ranking worsened when this feature ...
1,food_supply_kcal_cap_day_pct_change1,0.0080,0.0067,Validation ranking worsened when this feature ...
2,fat_supply_g_cap_day_change1,0.0074,0.0049,Validation ranking worsened when this feature ...
3,food_supply_kcal_cap_day,0.0070,0.0008,Validation ranking worsened when this feature ...
4,fat_supply_g_cap_day_pct_change1,0.0055,0.0063,Validation ranking worsened when this feature ...
5,protein_supply_g_cap_day_change1,0.0051,0.0050,Validation ranking worsened when this feature ...
6,food_supply_kcal_cap_day_rolling3_mean,0.0050,0.0018,Validation ranking worsened when this feature ...
7,food_supply_kcal_cap_day_lag1,0.0046,0.0015,Validation ranking worsened when this feature ...
8,food_supply_quantity_kg_cap_yr_pct_change1,0.0043,0.0063,Validation ranking worsened when this feature ...
9,Item Code,0.0038,0.0012,Validation ranking worsened when this feature ...



Selected-model audit summary:


,Result
Selected model,Temporally tuned random forest
Selected threshold,0.1200
Validation shortages,52
Validation shortages detected,37
Validation shortages missed,15
Validation false warnings,152
Validation recall %,71.1538
Validation precision %,19.5767
Validation F2,0.4660
Commodities evaluated,8



The selected model passed the pre-test interpretability and subgroup audit.
The locked test period remains unused.


In [14]:
# ---------------------------------------------------------
# Collect out-of-time forest predictions from 2015–2018
# ---------------------------------------------------------

out_of_time_rows = []

for fold_number, fold_values in (
    ensemble_temporal_prediction_store.items()
):
    validation_year = (
        fold_values["Validation year"]
    )

    y_fold = np.asarray(
        fold_values["y_true"],
        dtype=int,
    )

    probability_fold = np.asarray(
        fold_values[
            "Forest probability"
        ],
        dtype=float,
    )

    for actual_value, probability in zip(
        y_fold,
        probability_fold,
    ):
        out_of_time_rows.append(
            {
                "Predictor year": (
                    validation_year
                ),
                "Actual shortage": (
                    actual_value
                ),
                "Predicted probability": (
                    probability
                ),
                "Source": (
                    "2015–2018 expanding fold"
                ),
            }
        )

# ---------------------------------------------------------
# Add the 2019–2020 validation predictions
# ---------------------------------------------------------

for year, actual_value, probability in zip(
    validation_data["Year"].to_numpy(),
    y_validation.to_numpy(),
    tuned_rf_validation_probability,
):
    out_of_time_rows.append(
        {
            "Predictor year": int(year),
            "Actual shortage": int(
                actual_value
            ),
            "Predicted probability": float(
                probability
            ),
            "Source": (
                "2019–2020 main validation"
            ),
        }
    )

out_of_time_predictions = pd.DataFrame(
    out_of_time_rows
)

# ---------------------------------------------------------
# Select the final threshold from pooled out-of-time evidence
# ---------------------------------------------------------

pooled_threshold_rows = []

for threshold in threshold_grid:
    result, _ = evaluate_probabilities(
        y_true=out_of_time_predictions[
            "Actual shortage"
        ],
        predicted_probability=(
            out_of_time_predictions[
                "Predicted probability"
            ]
        ),
        threshold=threshold,
        model_name=(
            "Selected random forest"
        ),
        dataset_name=(
            "Pooled out-of-time development"
        ),
    )

    pooled_threshold_rows.append(
        result
    )

pooled_threshold_results = pd.DataFrame(
    pooled_threshold_rows
)

final_threshold_row = (
    pooled_threshold_results
    .sort_values(
        [
            "F2",
            "Recall",
            "Precision",
            "Threshold",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .iloc[0]
)

final_probability_threshold = float(
    final_threshold_row["Threshold"]
)

# ---------------------------------------------------------
# Compare the old and final thresholds
# ---------------------------------------------------------

threshold_lock_comparison_rows = []

for threshold_name, threshold in [
    (
        "Main-validation threshold",
        tuned_rf_selected_threshold,
    ),
    (
        "Pooled out-of-time threshold",
        final_probability_threshold,
    ),
]:
    result, _ = evaluate_probabilities(
        y_true=out_of_time_predictions[
            "Actual shortage"
        ],
        predicted_probability=(
            out_of_time_predictions[
                "Predicted probability"
            ]
        ),
        threshold=threshold,
        model_name=(
            "Selected random forest"
        ),
        dataset_name=(
            "Pooled out-of-time development"
        ),
    )

    threshold_lock_comparison_rows.append(
        {
            "Threshold source": (
                threshold_name
            ),
            "Threshold": threshold,
            "Observations": (
                result["Observations"]
            ),
            "Actual shortages": (
                result["Actual events"]
            ),
            "Warnings issued": (
                result["Predicted events"]
            ),
            "Shortages detected": (
                result["True positives"]
            ),
            "Shortages missed": (
                result["False negatives"]
            ),
            "False warnings": (
                result["False positives"]
            ),
            "Precision": (
                result["Precision"]
            ),
            "Recall": result["Recall"],
            "F1": result["F1"],
            "F2": result["F2"],
            "Balanced accuracy": (
                result[
                    "Balanced accuracy"
                ]
            ),
        }
    )

threshold_lock_comparison = pd.DataFrame(
    threshold_lock_comparison_rows
)

print(
    "Pooled out-of-time threshold comparison:"
)
display(threshold_lock_comparison)

# ---------------------------------------------------------
# Audit the locked threshold year by year
# ---------------------------------------------------------

locked_oof_probability = (
    out_of_time_predictions[
        "Predicted probability"
    ].to_numpy()
)

locked_oof_prediction = (
    locked_oof_probability
    >= final_probability_threshold
).astype(int)

out_of_time_predictions[
    "Predicted warning"
] = locked_oof_prediction

out_of_time_year_rows = []

for year, year_data in (
    out_of_time_predictions
    .groupby(
        "Predictor year",
        observed=True,
    )
):
    y_year = year_data[
        "Actual shortage"
    ].to_numpy(dtype=int)

    prediction_year = year_data[
        "Predicted warning"
    ].to_numpy(dtype=int)

    probability_year = year_data[
        "Predicted probability"
    ].to_numpy(dtype=float)

    tn, fp, fn, tp = confusion_matrix(
        y_year,
        prediction_year,
        labels=[0, 1],
    ).ravel()

    out_of_time_year_rows.append(
        {
            "Predictor year": year,
            "Observations": len(year_data),
            "Actual shortages": (
                int(y_year.sum())
            ),
            "Warnings issued": (
                int(
                    prediction_year.sum()
                )
            ),
            "Shortages detected": int(tp),
            "Shortages missed": int(fn),
            "False warnings": int(fp),
            "Precision": precision_score(
                y_year,
                prediction_year,
                zero_division=0,
            ),
            "Recall": recall_score(
                y_year,
                prediction_year,
                zero_division=0,
            ),
            "F2": fbeta_score(
                y_year,
                prediction_year,
                beta=2,
                zero_division=0,
            ),
            "PR-AUC": (
                average_precision_score(
                    y_year,
                    probability_year,
                )
            ),
        }
    )

out_of_time_year_summary = pd.DataFrame(
    out_of_time_year_rows
)

print(
    "\nFinal locked threshold across "
    "out-of-time development years:"
)
display(out_of_time_year_summary)

# ---------------------------------------------------------
# Refit the fixed model on all development data
# ---------------------------------------------------------

development_data = (
    pd.concat(
        [
            train_data,
            validation_data,
        ],
        axis=0,
    )
    .sort_values(
        [
            "Area",
            "Item Code",
            "Year",
        ]
    )
    .copy()
)

X_development = development_data[
    feature_columns
].copy()

y_development = development_data[
    target_column
].astype("int8").copy()

final_model = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor),
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=(
                    selected_rf_configuration[
                        "Maximum depth"
                    ]
                ),
                min_samples_leaf=(
                    selected_rf_configuration[
                        "Minimum leaf size"
                    ]
                ),
                max_features="sqrt",
                class_weight=None,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

final_model.fit(
    X_development,
    y_development,
)

# ---------------------------------------------------------
# Record the final pre-test lock
# ---------------------------------------------------------

final_model_lock_record = pd.Series(
    {
        "Model family": "Random forest",
        "Trees": 500,
        "Maximum depth": (
            "Unlimited"
            if selected_rf_configuration[
                "Maximum depth"
            ] is None
            else selected_rf_configuration[
                "Maximum depth"
            ]
        ),
        "Minimum observations per leaf": (
            selected_rf_configuration[
                "Minimum leaf size"
            ]
        ),
        "Class weighting": "None",
        "Development rows used for final fit": (
            len(X_development)
        ),
        "Development predictor year start": (
            development_data["Year"].min()
        ),
        "Development predictor year end": (
            development_data["Year"].max()
        ),
        "Development events": (
            y_development.sum()
        ),
        "Final probability threshold": (
            final_probability_threshold
        ),
        "Threshold evidence years": (
            "2015–2020 out-of-time predictions"
        ),
        "Threshold selection measure": (
            "Highest pooled F2"
        ),
        "Final predictors": (
            len(feature_columns)
        ),
        "Test outcomes inspected": False,
    },
    name="Locked setting",
).to_frame()

print("\nFinal pre-test model lock:")
display(final_model_lock_record)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert len(
    out_of_time_predictions
) == 1497

assert out_of_time_predictions[
    "Predictor year"
].min() == 2015

assert out_of_time_predictions[
    "Predictor year"
].max() == 2020

assert out_of_time_predictions[
    "Actual shortage"
].sum() == 155

assert (
    0.01
    <= final_probability_threshold
    <= 0.99
)

assert len(X_development) == 2750
assert y_development.sum() == 283

assert development_data[
    "Year"
].min() == 2010

assert development_data[
    "Year"
].max() == 2020

final_fitted_clipper = (
    final_model
    .named_steps[
        "preprocessor"
    ]
    .named_transformers_[
        "continuous_numeric"
    ]
    .named_steps[
        "clipper"
    ]
)

assert (
    final_fitted_clipper.fit_row_count_
    == 2750
)

assert (
    final_model_lock_record.loc[
        "Test outcomes inspected",
        "Locked setting",
    ]
    is False
)

print(
    "\nThe final model settings and probability "
    "threshold were locked before test evaluation."
)

print(
    "The test period remains unopened."
)

Pooled out-of-time threshold comparison:


,Threshold source,Threshold,Observations,Actual shortages,Warnings issued,Shortages detected,Shortages missed,False warnings,Precision,Recall,F1,F2,Balanced accuracy
0,Main-validation threshold,0.1200,1497,155,499,100,55,399,0.2004,0.6452,0.3058,0.4468,0.6739
1,Pooled out-of-time threshold,0.0700,1497,155,788,126,29,662,0.1599,0.8129,0.2672,0.4474,0.6598



Final locked threshold across out-of-time development years:


,Predictor year,Observations,Actual shortages,Warnings issued,Shortages detected,Shortages missed,False warnings,Precision,Recall,F2,PR-AUC
0,2015,249,29,123,22,7,101,0.1789,0.7586,0.4603,0.3259
1,2016,249,14,122,11,3,111,0.0902,0.7857,0.3090,0.1980
2,2017,249,26,127,23,3,104,0.1811,0.8846,0.4978,0.4990
3,2018,250,34,119,26,8,93,0.2185,0.7647,0.5098,0.3039
4,2019,246,24,143,19,5,124,0.1329,0.7917,0.3975,0.3105
5,2020,254,28,154,25,3,129,0.1623,0.8929,0.4699,0.2718



Final pre-test model lock:


,Locked setting
Model family,Random forest
Trees,500
Maximum depth,Unlimited
Minimum observations per leaf,3
Class weighting,None
Development rows used for final fit,2750
Development predictor year start,2010
Development predictor year end,2020
Development events,283
Final probability threshold,0.0700



The final model settings and probability threshold were locked before test evaluation.
The test period remains unopened.


In [15]:
# ---------------------------------------------------------
# Define the near-optimal F2 range
# ---------------------------------------------------------

maximum_pooled_f2 = (
    pooled_threshold_results[
        "F2"
    ].max()
)

near_optimal_f2_boundary = (
    0.99 * maximum_pooled_f2
)

near_optimal_thresholds = (
    pooled_threshold_results.loc[
        pooled_threshold_results[
            "F2"
        ].ge(
            near_optimal_f2_boundary
        )
    ]
    .sort_values(
        [
            "Precision",
            "Predicted events",
            "Recall",
            "Threshold",
        ],
        ascending=[
            False,
            True,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

print(
    "Near-optimal pooled F2 thresholds "
    "(within 99% of the maximum):"
)

display(
    near_optimal_thresholds[
        [
            "Threshold",
            "Predicted events",
            "True positives",
            "False positives",
            "False negatives",
            "Precision",
            "Recall",
            "F1",
            "F2",
            "Balanced accuracy",
        ]
    ]
)

# ---------------------------------------------------------
# Select the most precise near-optimal threshold
# ---------------------------------------------------------

robust_threshold_row = (
    near_optimal_thresholds.iloc[0]
)

robust_final_probability_threshold = float(
    robust_threshold_row[
        "Threshold"
    ]
)

robust_threshold_comparison_rows = []

for threshold_name, threshold in [
    (
        "Exact maximum-F2 threshold",
        final_probability_threshold,
    ),
    (
        "Most precise near-optimal threshold",
        robust_final_probability_threshold,
    ),
]:
    result, _ = evaluate_probabilities(
        y_true=out_of_time_predictions[
            "Actual shortage"
        ],
        predicted_probability=(
            out_of_time_predictions[
                "Predicted probability"
            ]
        ),
        threshold=threshold,
        model_name=(
            "Selected random forest"
        ),
        dataset_name=(
            "Pooled out-of-time development"
        ),
    )

    robust_threshold_comparison_rows.append(
        {
            "Threshold rule": threshold_name,
            "Threshold": threshold,
            "Warnings issued": (
                result["Predicted events"]
            ),
            "Shortages detected": (
                result["True positives"]
            ),
            "Shortages missed": (
                result["False negatives"]
            ),
            "False warnings": (
                result["False positives"]
            ),
            "Precision": (
                result["Precision"]
            ),
            "Recall": result["Recall"],
            "F1": result["F1"],
            "F2": result["F2"],
            "Balanced accuracy": (
                result[
                    "Balanced accuracy"
                ]
            ),
        }
    )

robust_threshold_comparison = pd.DataFrame(
    robust_threshold_comparison_rows
)

print(
    "\nExact optimum versus robust "
    "near-optimal threshold:"
)
display(robust_threshold_comparison)

# ---------------------------------------------------------
# Lock the robust threshold
# ---------------------------------------------------------

final_probability_threshold = (
    robust_final_probability_threshold
)

final_model_lock_record.loc[
    "Final probability threshold",
    "Locked setting",
] = final_probability_threshold

final_model_lock_record.loc[
    "Threshold selection measure",
    "Locked setting",
] = (
    "Highest precision among thresholds "
    "within 99% of maximum pooled F2"
)

final_model_lock_record.loc[
    "Test outcomes inspected",
    "Locked setting",
] = False

print("\nRevised final pre-test model lock:")
display(final_model_lock_record)

# ---------------------------------------------------------
# Formal validations
# ---------------------------------------------------------

assert (
    robust_threshold_row["F2"]
    >= near_optimal_f2_boundary
)

assert (
    robust_threshold_row[
        "Precision"
    ]
    == near_optimal_thresholds[
        "Precision"
    ].max()
)

assert (
    0.01
    <= final_probability_threshold
    <= 0.99
)

assert (
    final_model_lock_record.loc[
        "Test outcomes inspected",
        "Locked setting",
    ]
    is False
)

print(
    "\nThe robust probability threshold was "
    "locked using development data only."
)

print(
    "The test period remains unopened."
)

Near-optimal pooled F2 thresholds (within 99% of the maximum):


,Threshold,Predicted events,True positives,False positives,False negatives,Precision,Recall,F1,F2,Balanced accuracy
0,0.1300,452,95,357,60,0.2102,0.6129,0.3130,0.4431,0.6734
1,0.1200,499,100,399,55,0.2004,0.6452,0.3058,0.4468,0.6739
2,0.0900,644,113,531,42,0.1755,0.7290,0.2829,0.4470,0.6667
3,0.0700,788,126,662,29,0.1599,0.8129,0.2672,0.4474,0.6598



Exact optimum versus robust near-optimal threshold:


,Threshold rule,Threshold,Warnings issued,Shortages detected,Shortages missed,False warnings,Precision,Recall,F1,F2,Balanced accuracy
0,Exact maximum-F2 threshold,0.0700,788,126,29,662,0.1599,0.8129,0.2672,0.4474,0.6598
1,Most precise near-optimal threshold,0.1300,452,95,60,357,0.2102,0.6129,0.3130,0.4431,0.6734



Revised final pre-test model lock:


,Locked setting
Model family,Random forest
Trees,500
Maximum depth,Unlimited
Minimum observations per leaf,3
Class weighting,None
Development rows used for final fit,2750
Development predictor year start,2010
Development predictor year end,2020
Development events,283
Final probability threshold,0.1300



The robust probability threshold was locked using development data only.
The test period remains unopened.


In [16]:
# ============================================================
# FINAL LOCKED-TEST EVALUATION
# Predictor years: 2021–2022
# Outcome years:   2022–2023
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
)

# ------------------------------------------------------------
# 1. Confirm that the pre-test decisions remain locked
# ------------------------------------------------------------

assert final_probability_threshold == 0.13
assert len(locked_test_data) == 498
assert locked_test_data["Year"].min() == 2021
assert locked_test_data["Year"].max() == 2022
assert locked_test_data["target_year"].min() == 2022
assert locked_test_data["target_year"].max() == 2023

locked_model_description = {
    "Model family": "Random forest",
    "Trees": 500,
    "Maximum depth": "Unlimited",
    "Minimum observations per leaf": 3,
    "Class weighting": "None",
    "Final probability threshold": final_probability_threshold,
    "Predictors": 129,
}

print("Locked model settings:")
display(
    pd.DataFrame(
        locked_model_description.items(),
        columns=["Locked setting", "Value"],
    )
)

# ------------------------------------------------------------
# 2. Prepare the locked test predictors
# ------------------------------------------------------------

# X_test_locked should contain the same 129 raw predictors used
# during model development.
assert X_test_locked.shape[0] == 498
assert X_test_locked.shape[1] == 129

# Support either:
# A. a complete modelling pipeline, or
# B. a separate fitted preprocessor and fitted random forest.
if hasattr(final_model, "named_steps"):
    final_test_probability = final_model.predict_proba(
        X_test_locked
    )[:, 1]

else:
    if "final_preprocessor" not in globals():
        raise NameError(
            "The fitted final_preprocessor was not found. "
            "Do not refit anything. Restore the final_preprocessor "
            "created in the previous model-lock cell."
        )

    X_test_processed = final_preprocessor.transform(X_test_locked)

    assert X_test_processed.shape[0] == 498
    assert np.isfinite(X_test_processed).all()

    final_test_probability = final_model.predict_proba(
        X_test_processed
    )[:, 1]

assert len(final_test_probability) == 498
assert np.isfinite(final_test_probability).all()
assert (
    (final_test_probability >= 0)
    & (final_test_probability <= 1)
).all()

# ------------------------------------------------------------
# 3. Open the locked test outcomes for the first and only time
# ------------------------------------------------------------

y_test = (
    locked_test_data["shortage_next_year"]
    .astype(int)
    .to_numpy()
)

assert set(np.unique(y_test)).issubset({0, 1})

final_test_warning = (
    final_test_probability >= final_probability_threshold
).astype(int)

# ------------------------------------------------------------
# 4. Calculate the final test results
# ------------------------------------------------------------

tn, fp, fn, tp = confusion_matrix(
    y_test,
    final_test_warning,
    labels=[0, 1],
).ravel()

test_metrics = pd.DataFrame(
    [{
        "Model": "Locked random forest",
        "Dataset": "Final locked test",
        "Threshold": final_probability_threshold,
        "Observations": len(y_test),
        "Actual shortages": int(y_test.sum()),
        "Warnings issued": int(final_test_warning.sum()),
        "True negatives": int(tn),
        "False positives": int(fp),
        "False negatives": int(fn),
        "True positives": int(tp),
        "Accuracy": accuracy_score(
            y_test, final_test_warning
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_test, final_test_warning
        ),
        "Precision": precision_score(
            y_test,
            final_test_warning,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_test,
            final_test_warning,
            zero_division=0,
        ),
        "Specificity": (
            tn / (tn + fp) if (tn + fp) else np.nan
        ),
        "F1": f1_score(
            y_test,
            final_test_warning,
            zero_division=0,
        ),
        "F2": fbeta_score(
            y_test,
            final_test_warning,
            beta=2,
            zero_division=0,
        ),
        "ROC-AUC": roc_auc_score(
            y_test,
            final_test_probability,
        ),
        "PR-AUC": average_precision_score(
            y_test,
            final_test_probability,
        ),
        "Brier score": brier_score_loss(
            y_test,
            final_test_probability,
        ),
    }]
)

print("\nFinal locked-test performance:")
display(
    test_metrics.style.format({
        "Threshold": "{:.4f}",
        "Accuracy": "{:.4f}",
        "Balanced accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "Specificity": "{:.4f}",
        "F1": "{:.4f}",
        "F2": "{:.4f}",
        "ROC-AUC": "{:.4f}",
        "PR-AUC": "{:.4f}",
        "Brier score": "{:.4f}",
    })
)

# ------------------------------------------------------------
# 5. Plain-language warning summary
# ------------------------------------------------------------

plain_language_summary = pd.DataFrame(
    {
        "Result": [
            "Country-commodity observations tested",
            "Real next-year shortages",
            "Shortages detected",
            "Shortages missed",
            "Warnings issued",
            "False warnings",
            "Correctly rejected non-shortages",
            "Percentage of shortages detected",
            "Percentage of warnings that were correct",
        ],
        "Value": [
            len(y_test),
            int(y_test.sum()),
            int(tp),
            int(fn),
            int(final_test_warning.sum()),
            int(fp),
            int(tn),
            100 * tp / (tp + fn) if (tp + fn) else np.nan,
            100 * tp / (tp + fp) if (tp + fp) else np.nan,
        ],
    }
)

print("\nFinal test results in plain language:")
display(
    plain_language_summary.style.format(
        {"Value": "{:.2f}"}
    )
)

# ------------------------------------------------------------
# 6. Confusion matrix
# ------------------------------------------------------------

test_confusion_matrix = pd.DataFrame(
    [
        [tn, fp],
        [fn, tp],
    ],
    index=pd.Index(
        ["No shortage", "Shortage"],
        name="Actual outcome",
    ),
    columns=[
        "Predicted no shortage",
        "Predicted shortage",
    ],
)

print("\nFinal test confusion matrix:")
display(test_confusion_matrix)

# ------------------------------------------------------------
# 7. Save row-level test predictions in memory
# ------------------------------------------------------------

test_predictions = locked_test_data[
    [
        "Area",
        "Item Code",
        "Item",
        "Year",
        "target_year",
        "shortage_next_year",
    ]
].copy()

test_predictions[
    "predicted_shortage_probability"
] = final_test_probability

test_predictions[
    "predicted_shortage_warning"
] = final_test_warning

test_predictions["prediction_status"] = np.select(
    [
        (test_predictions["shortage_next_year"] == 1)
        & (test_predictions["predicted_shortage_warning"] == 1),

        (test_predictions["shortage_next_year"] == 0)
        & (test_predictions["predicted_shortage_warning"] == 1),

        (test_predictions["shortage_next_year"] == 1)
        & (test_predictions["predicted_shortage_warning"] == 0),
    ],
    [
        "Shortage detected",
        "False warning",
        "Shortage missed",
    ],
    default="Correct non-shortage",
)

# ------------------------------------------------------------
# 8. Performance by test predictor year
# ------------------------------------------------------------

def summarise_test_group(group):
    actual = group["shortage_next_year"].astype(int)
    warning = group["predicted_shortage_warning"].astype(int)
    probability = group["predicted_shortage_probability"]

    group_tn, group_fp, group_fn, group_tp = confusion_matrix(
        actual,
        warning,
        labels=[0, 1],
    ).ravel()

    return pd.Series(
        {
            "Observations": len(group),
            "Actual shortages": int(actual.sum()),
            "Warnings issued": int(warning.sum()),
            "Shortages detected": int(group_tp),
            "Shortages missed": int(group_fn),
            "False warnings": int(group_fp),
            "Precision": precision_score(
                actual, warning, zero_division=0
            ),
            "Recall": recall_score(
                actual, warning, zero_division=0
            ),
            "F2": fbeta_score(
                actual,
                warning,
                beta=2,
                zero_division=0,
            ),
            "ROC-AUC": (
                roc_auc_score(actual, probability)
                if actual.nunique() == 2
                else np.nan
            ),
            "PR-AUC": (
                average_precision_score(actual, probability)
                if actual.sum() > 0
                else np.nan
            ),
        }
    )

test_performance_by_year = (
    test_predictions
    .groupby("Year", observed=True)
    .apply(summarise_test_group, include_groups=False)
    .reset_index()
)

print("\nFinal test performance by predictor year:")
display(
    test_performance_by_year.style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F2": "{:.4f}",
        "ROC-AUC": "{:.4f}",
        "PR-AUC": "{:.4f}",
    })
)

# ------------------------------------------------------------
# 9. Performance by commodity
# ------------------------------------------------------------

test_performance_by_commodity = (
    test_predictions
    .groupby("Item", observed=True)
    .apply(summarise_test_group, include_groups=False)
    .reset_index()
    .sort_values(
        ["Actual shortages", "Recall"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

print("\nFinal test performance by commodity:")
display(
    test_performance_by_commodity.style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F2": "{:.4f}",
        "ROC-AUC": "{:.4f}",
        "PR-AUC": "{:.4f}",
    })
)

# ------------------------------------------------------------
# 10. Twenty highest predicted test risks
# ------------------------------------------------------------

highest_test_risks = (
    test_predictions
    .sort_values(
        "predicted_shortage_probability",
        ascending=False,
    )
    .head(20)
    .reset_index(drop=True)
)

print("\nTwenty highest predicted risks in the locked test period:")
display(
    highest_test_risks.style.format(
        {"predicted_shortage_probability": "{:.4f}"}
    )
)

print(
    "\nThe locked test period was evaluated exactly once.\n"
    "The model family, fitted model and 0.13 threshold were not "
    "changed after the test outcomes were opened."
)

Locked model settings:


,Locked setting,Value
0,Model family,Random forest
1,Trees,500
2,Maximum depth,Unlimited
3,Minimum observations per leaf,3
4,Class weighting,None
5,Final probability threshold,0.1300
6,Predictors,129



Final locked-test performance:


,Model,Dataset,Threshold,Observations,Actual shortages,Warnings issued,True negatives,False positives,False negatives,True positives,Accuracy,Balanced accuracy,Precision,Recall,Specificity,F1,F2,ROC-AUC,PR-AUC,Brier score
0,Locked random forest,Final locked test,0.1300,498,51,159,321,126,18,33,0.7108,0.6826,0.2075,0.6471,0.7181,0.3143,0.4545,0.7597,0.3071,0.0823



Final test results in plain language:


...


Final test performance by commodity:


,Item,Observations,Actual shortages,Warnings issued,Shortages detected,Shortages missed,False warnings,Precision,Recall,F2,ROC-AUC,PR-AUC
0,Wheat and products,86.000000,12.000000,18.000000,4.000000,8.000000,14.000000,0.2222,0.3333,0.3030,0.7005,0.2600
1,Groundnuts,72.000000,11.000000,42.000000,9.000000,2.000000,33.000000,0.2143,0.8182,0.5233,0.7765,0.4656
2,Sorghum and products,50.000000,11.000000,24.000000,9.000000,2.000000,15.000000,0.3750,0.8182,0.6618,0.8065,0.5495
3,Maize and products,81.000000,6.000000,22.000000,5.000000,1.000000,17.000000,0.2273,0.8333,0.5435,0.8267,0.5002
4,Rice and products,86.000000,6.000000,27.000000,4.000000,2.000000,23.000000,0.1481,0.6667,0.3922,0.7208,0.3605
5,Millet and products,43.000000,3.000000,14.000000,2.000000,1.000000,12.000000,0.1429,0.6667,0.3846,0.6250,0.1186
6,Cassava and products,55.000000,2.000000,8.000000,0.000000,2.000000,8.000000,0.0000,0.0000,0.0000,0.5849,0.0635
7,Yams,25.000000,0.000000,4.000000,0.000000,0.000000,4.000000,0.0000,0.0000,0.0000,nan,nan



Twenty highest predicted risks in the locked test period:


,Area,Item Code,Item,Year,target_year,shortage_next_year,predicted_shortage_probability,predicted_shortage_warning,prediction_status
0,Malawi,2552,Groundnuts,2022,2023,0,0.7312,1,False warning
1,Rwanda,2511,Wheat and products,2021,2022,0,0.6899,1,False warning
2,Uganda,2518,Sorghum and products,2021,2022,0,0.6017,1,False warning
3,Kenya,2518,Sorghum and products,2022,2023,0,0.5947,1,False warning
4,Botswana,2518,Sorghum and products,2022,2023,1,0.5903,1,Shortage detected
5,Lesotho,2518,Sorghum and products,2021,2022,1,0.5708,1,Shortage detected
6,Uganda,2552,Groundnuts,2022,2023,1,0.5598,1,Shortage detected
7,Mauritania,2518,Sorghum and products,2022,2023,1,0.5477,1,Shortage detected
8,Gambia,2514,Maize and products,2021,2022,1,0.5129,1,Shortage detected
9,Uganda,2517,Millet and products,2022,2023,0,0.5039,1,False warning



The locked test period was evaluated exactly once.
The model family, fitted model and 0.13 threshold were not changed after the test outcomes were opened.


In [17]:
# ============================================================
# FINAL TEST UNCERTAINTY AND RISK-CALIBRATION AUDIT
# This cell does not retrain the model or change the threshold.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    balanced_accuracy_score,
    precision_score,
    recall_score,
    fbeta_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
)

# ------------------------------------------------------------
# 1. Reconfirm the locked decision
# ------------------------------------------------------------

assert final_probability_threshold == 0.13
assert len(test_predictions) == 498
assert len(y_test) == 498

original_test_probabilities = (
    test_predictions[
        "predicted_shortage_probability"
    ].to_numpy().copy()
)

original_test_warnings = (
    test_predictions[
        "predicted_shortage_warning"
    ].to_numpy().copy()
)

assert np.array_equal(
    original_test_warnings,
    (
        original_test_probabilities
        >= final_probability_threshold
    ).astype(int),
)

# ------------------------------------------------------------
# 2. Compare like-for-like development and test performance
# ------------------------------------------------------------

development_test_comparison = pd.DataFrame(
    [
        {
            "Evaluation period":
                "Out-of-time development years 2015–2020",
            "Threshold": 0.13,
            "Observations": 1497,
            "Actual shortages": 155,
            "Warnings issued": 452,
            "Precision": 0.2102,
            "Recall": 0.6129,
            "F2": 0.4431,
            "Balanced accuracy": 0.6734,
        },
        {
            "Evaluation period":
                "Final locked test years 2021–2022",
            "Threshold": final_probability_threshold,
            "Observations": len(y_test),
            "Actual shortages": int(y_test.sum()),
            "Warnings issued":
                int(original_test_warnings.sum()),
            "Precision": precision_score(
                y_test,
                original_test_warnings,
                zero_division=0,
            ),
            "Recall": recall_score(
                y_test,
                original_test_warnings,
                zero_division=0,
            ),
            "F2": fbeta_score(
                y_test,
                original_test_warnings,
                beta=2,
                zero_division=0,
            ),
            "Balanced accuracy":
                balanced_accuracy_score(
                    y_test,
                    original_test_warnings,
                ),
        },
    ]
)

print("Development versus final-test performance:")
display(
    development_test_comparison.style.format({
        "Threshold": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F2": "{:.4f}",
        "Balanced accuracy": "{:.4f}",
    })
)

# ------------------------------------------------------------
# 3. Calculate useful test-period reference measures
# ------------------------------------------------------------

test_event_rate = y_test.mean()
test_pr_auc = average_precision_score(
    y_test,
    original_test_probabilities,
)
test_pr_auc_lift = test_pr_auc / test_event_rate

tn, fp, fn, tp = confusion_matrix(
    y_test,
    original_test_warnings,
    labels=[0, 1],
).ravel()

negative_predictive_value = (
    tn / (tn + fn)
    if (tn + fn) > 0
    else np.nan
)

reference_summary = pd.DataFrame(
    {
        "Measure": [
            "Overall shortage rate",
            "PR-AUC no-skill reference",
            "Model PR-AUC",
            "PR-AUC lift above the no-skill reference",
            "Warning accuracy",
            "Non-warning accuracy",
        ],
        "Value": [
            test_event_rate,
            test_event_rate,
            test_pr_auc,
            test_pr_auc_lift,
            tp / (tp + fp),
            negative_predictive_value,
        ],
        "Plain-English meaning": [
            "About this share of all tested observations became shortages.",
            "A model with no useful ranking ability would be expected around this value.",
            "How well the model prioritised shortages when shortages were uncommon.",
            "How many times the model's PR-AUC exceeded the no-skill reference.",
            "Among issued warnings, the share that became shortages.",
            "Among non-warning cases, the share that did not become shortages.",
        ],
    }
)

print("\nFinal-test reference measures:")
display(
    reference_summary.style.format(
        {"Value": "{:.4f}"}
    )
)

# ------------------------------------------------------------
# 4. Cluster bootstrap uncertainty intervals
# ------------------------------------------------------------
# A confidence interval gives a plausible range around a result.
#
# We resample whole country-commodity pairs rather than isolated
# rows. This respects the fact that the two years belonging to
# one country-commodity pair may be related.

bootstrap_data = test_predictions.copy()

bootstrap_data["pair_id"] = (
    bootstrap_data["Area"].astype(str)
    + " | "
    + bootstrap_data["Item Code"].astype(str)
)

unique_pairs = bootstrap_data["pair_id"].unique()

bootstrap_iterations = 2000
random_generator = np.random.default_rng(20260824)

bootstrap_results = []

for iteration in range(bootstrap_iterations):

    sampled_pairs = random_generator.choice(
        unique_pairs,
        size=len(unique_pairs),
        replace=True,
    )

    sampled_frames = [
        bootstrap_data.loc[
            bootstrap_data["pair_id"] == pair
        ]
        for pair in sampled_pairs
    ]

    sample = pd.concat(
        sampled_frames,
        ignore_index=True,
    )

    actual = (
        sample["shortage_next_year"]
        .astype(int)
        .to_numpy()
    )

    warning = (
        sample["predicted_shortage_warning"]
        .astype(int)
        .to_numpy()
    )

    probability = (
        sample["predicted_shortage_probability"]
        .to_numpy()
    )

    sample_tn, sample_fp, sample_fn, sample_tp = (
        confusion_matrix(
            actual,
            warning,
            labels=[0, 1],
        ).ravel()
    )

    # ROC-AUC requires both outcome classes.
    if np.unique(actual).size < 2:
        continue

    bootstrap_results.append(
        {
            "Precision": precision_score(
                actual,
                warning,
                zero_division=0,
            ),
            "Recall": recall_score(
                actual,
                warning,
                zero_division=0,
            ),
            "Specificity": (
                sample_tn / (sample_tn + sample_fp)
                if (sample_tn + sample_fp) > 0
                else np.nan
            ),
            "F2": fbeta_score(
                actual,
                warning,
                beta=2,
                zero_division=0,
            ),
            "Balanced accuracy":
                balanced_accuracy_score(
                    actual,
                    warning,
                ),
            "ROC-AUC": roc_auc_score(
                actual,
                probability,
            ),
            "PR-AUC": average_precision_score(
                actual,
                probability,
            ),
            "Brier score": brier_score_loss(
                actual,
                probability,
            ),
        }
    )

bootstrap_results = pd.DataFrame(bootstrap_results)

point_estimates = {
    "Precision": precision_score(
        y_test,
        original_test_warnings,
        zero_division=0,
    ),
    "Recall": recall_score(
        y_test,
        original_test_warnings,
        zero_division=0,
    ),
    "Specificity": tn / (tn + fp),
    "F2": fbeta_score(
        y_test,
        original_test_warnings,
        beta=2,
        zero_division=0,
    ),
    "Balanced accuracy": balanced_accuracy_score(
        y_test,
        original_test_warnings,
    ),
    "ROC-AUC": roc_auc_score(
        y_test,
        original_test_probabilities,
    ),
    "PR-AUC": average_precision_score(
        y_test,
        original_test_probabilities,
    ),
    "Brier score": brier_score_loss(
        y_test,
        original_test_probabilities,
    ),
}

uncertainty_rows = []

for measure, point_estimate in point_estimates.items():
    uncertainty_rows.append(
        {
            "Measure": measure,
            "Test estimate": point_estimate,
            "Lower 95% confidence boundary":
                bootstrap_results[measure].quantile(0.025),
            "Upper 95% confidence boundary":
                bootstrap_results[measure].quantile(0.975),
        }
    )

test_uncertainty_summary = pd.DataFrame(
    uncertainty_rows
)

print(
    "\nFinal-test estimates with 95% confidence intervals:"
)
display(
    test_uncertainty_summary.style.format({
        "Test estimate": "{:.4f}",
        "Lower 95% confidence boundary": "{:.4f}",
        "Upper 95% confidence boundary": "{:.4f}",
    })
)

# ------------------------------------------------------------
# 5. Descriptive risk bands
# ------------------------------------------------------------
# These bands describe the already-produced probabilities.
# They are not new thresholds and will not alter predictions.

risk_band_boundaries = [
    0.00,
    0.05,
    0.13,
    0.30,
    1.000001,
]

risk_band_labels = [
    "Very low: below 5%",
    "Watch: 5% to below 13%",
    "Warning: 13% to below 30%",
    "High warning: 30% or more",
]

test_predictions["risk_band"] = pd.cut(
    test_predictions[
        "predicted_shortage_probability"
    ],
    bins=risk_band_boundaries,
    labels=risk_band_labels,
    right=False,
    include_lowest=True,
)

test_risk_band_summary = (
    test_predictions
    .groupby(
        "risk_band",
        observed=False,
    )
    .agg(
        Observations=(
            "shortage_next_year",
            "size",
        ),
        Actual_shortages=(
            "shortage_next_year",
            "sum",
        ),
        Average_predicted_probability=(
            "predicted_shortage_probability",
            "mean",
        ),
        Minimum_predicted_probability=(
            "predicted_shortage_probability",
            "min",
        ),
        Maximum_predicted_probability=(
            "predicted_shortage_probability",
            "max",
        ),
    )
    .reset_index()
)

test_risk_band_summary[
    "Observed shortage rate"
] = (
    test_risk_band_summary["Actual_shortages"]
    / test_risk_band_summary["Observations"]
)

print("\nObserved outcomes within each risk band:")
display(
    test_risk_band_summary.style.format({
        "Average_predicted_probability": "{:.4f}",
        "Minimum_predicted_probability": "{:.4f}",
        "Maximum predicted probability": "{:.4f}",
        "Observed shortage rate": "{:.4f}",
    })
)

# ------------------------------------------------------------
# 6. Examine the missed shortages
# ------------------------------------------------------------

missed_test_shortages = (
    test_predictions.loc[
        test_predictions["prediction_status"]
        == "Shortage missed"
    ]
    .sort_values(
        "predicted_shortage_probability",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("\nAll shortages missed by the locked model:")
display(
    missed_test_shortages[
        [
            "Area",
            "Item Code",
            "Item",
            "Year",
            "target_year",
            "predicted_shortage_probability",
            "risk_band",
        ]
    ].style.format(
        {"predicted_shortage_probability": "{:.4f}"}
    )
)

# ------------------------------------------------------------
# 7. Final integrity checks
# ------------------------------------------------------------

assert final_probability_threshold == 0.13

assert np.array_equal(
    original_test_probabilities,
    test_predictions[
        "predicted_shortage_probability"
    ].to_numpy(),
)

assert np.array_equal(
    original_test_warnings,
    test_predictions[
        "predicted_shortage_warning"
    ].to_numpy(),
)

print(
    "\nFinal uncertainty and calibration audit completed.\n"
    "The fitted model, predicted probabilities, warning "
    "decisions and 0.13 threshold remained unchanged."
)

Development versus final-test performance:


,Evaluation period,Threshold,Observations,Actual shortages,Warnings issued,Precision,Recall,F2,Balanced accuracy
0,Out-of-time development years 2015–2020,0.1300,1497,155,452,0.2102,0.6129,0.4431,0.6734
1,Final locked test years 2021–2022,0.1300,498,51,159,0.2075,0.6471,0.4545,0.6826



Final-test reference measures:


,Measure,Value,Plain-English meaning
0,Overall shortage rate,0.1024,About this share of all tested observations became shortages.
1,PR-AUC no-skill reference,0.1024,A model with no useful ranking ability would be expected around this value.
2,Model PR-AUC,0.3071,How well the model prioritised shortages when shortages were uncommon.
3,PR-AUC lift above the no-skill reference,2.9983,How many times the model's PR-AUC exceeded the no-skill reference.
4,Warning accuracy,0.2075,"Among issued warnings, the share that became shortages."
5,Non-warning accuracy,0.9469,"Among non-warning cases, the share that did not become shortages."



Final-test estimates with 95% confidence intervals:


...


Observed outcomes within each risk band:


,risk_band,Observations,Actual_shortages,Average_predicted_probability,Minimum_predicted_probability,Maximum_predicted_probability,Observed shortage rate
0,Very low: below 5%,162,4,0.0294,0.0022,0.049896,0.0247
1,Watch: 5% to below 13%,177,14,0.0846,0.0508,0.129510,0.0791
2,Warning: 13% to below 30%,118,15,0.1902,0.1303,0.299027,0.1271
3,High warning: 30% or more,41,18,0.4360,0.3012,0.731223,0.4390



All shortages missed by the locked model:


,Area,Item Code,Item,Year,target_year,predicted_shortage_probability,risk_band
0,Guinea-Bissau,2511,Wheat and products,2022,2023,0.1157,Watch: 5% to below 13%
1,Liberia,2511,Wheat and products,2021,2022,0.1155,Watch: 5% to below 13%
2,Ghana,2511,Wheat and products,2022,2023,0.1107,Watch: 5% to below 13%
3,Mozambique,2511,Wheat and products,2021,2022,0.1098,Watch: 5% to below 13%
4,Liberia,2511,Wheat and products,2022,2023,0.1070,Watch: 5% to below 13%
5,United Republic of Tanzania,2518,Sorghum and products,2021,2022,0.0940,Watch: 5% to below 13%
6,Uganda,2532,Cassava and products,2022,2023,0.0782,Watch: 5% to below 13%
7,Democratic Republic of the Congo,2552,Groundnuts,2022,2023,0.0724,Watch: 5% to below 13%
8,Mozambique,2552,Groundnuts,2021,2022,0.0715,Watch: 5% to below 13%
9,Kenya,2511,Wheat and products,2021,2022,0.0706,Watch: 5% to below 13%



Final uncertainty and calibration audit completed.
The fitted model, predicted probabilities, warning decisions and 0.13 threshold remained unchanged.


In [18]:
# ============================================================
# SAVE THE FINAL LOCKED-TEST EVALUATION EVIDENCE
# This saves results only. It does not retrain or alter the model.
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np
import sklearn

model_output_directory = Path(
    "/Users/adewale/Documents/food_security_predictor/"
    "models/africa_first"
)

model_output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# 1. Create a final evaluation decision record
# ------------------------------------------------------------

final_evaluation_record = pd.DataFrame(
    {
        "Decision component": [
            "Model family",
            "Trees",
            "Maximum depth",
            "Minimum observations per leaf",
            "Class weighting",
            "Final probability threshold",
            "Threshold evidence",
            "Threshold selection rule",
            "Development predictor years",
            "Final test predictor years",
            "Final test outcome years",
            "Final predictors",
            "Test observations",
            "Test shortages",
            "Test warnings",
            "Test shortages detected",
            "Test shortages missed",
            "Test false warnings",
            "Model changed after test",
            "Threshold changed after test",
            "Intended use",
        ],
        "Decision": [
            "Random forest",
            500,
            "Unlimited",
            3,
            "None",
            0.13,
            "Out-of-time development predictions from 2015–2020",
            (
                "Highest precision among thresholds within "
                "99% of the maximum pooled development F2"
            ),
            "2010–2020",
            "2021–2022",
            "2022–2023",
            129,
            498,
            51,
            159,
            33,
            18,
            126,
            False,
            False,
            (
                "Early-warning screening and prioritisation; "
                "not an automatic shortage declaration"
            ),
        ],
    }
)

# ------------------------------------------------------------
# 2. Save the evaluation tables
# ------------------------------------------------------------

output_tables = {
    "shortage_final_test_metrics.csv":
        test_metrics,

    "shortage_final_test_uncertainty.csv":
        test_uncertainty_summary,

    "shortage_final_test_risk_bands.csv":
        test_risk_band_summary,

    "shortage_final_test_by_year.csv":
        test_performance_by_year,

    "shortage_final_test_by_commodity.csv":
        test_performance_by_commodity,

    "shortage_development_test_comparison.csv":
        development_test_comparison,

    "shortage_final_evaluation_record.csv":
        final_evaluation_record,

    "shortage_final_test_missed_events.csv":
        missed_test_shortages,
}

saved_file_records = []

for filename, table in output_tables.items():

    output_path = model_output_directory / filename

    table.to_csv(
        output_path,
        index=False,
    )

    saved_file_records.append(
        {
            "Output": filename,
            "Rows": len(table),
            "Columns": len(table.columns),
            "File size MB":
                output_path.stat().st_size / (1024 ** 2),
        }
    )

# ------------------------------------------------------------
# 3. Save all row-level test predictions
# ------------------------------------------------------------

test_predictions_path = (
    model_output_directory
    / "shortage_final_test_predictions.parquet"
)

test_predictions.to_parquet(
    test_predictions_path,
    index=False,
)

saved_file_records.append(
    {
        "Output":
            "shortage_final_test_predictions.parquet",
        "Rows": len(test_predictions),
        "Columns": len(test_predictions.columns),
        "File size MB":
            test_predictions_path.stat().st_size
            / (1024 ** 2),
    }
)

# ------------------------------------------------------------
# 4. Save machine-readable evaluation metadata
# ------------------------------------------------------------

final_evaluation_metadata = {
    "project":
        "Africa-first food supply shortage predictor",

    "model_family":
        "Random forest",

    "model_settings": {
        "number_of_trees": 500,
        "maximum_depth": None,
        "minimum_observations_per_leaf": 3,
        "features_considered_per_split": "square root",
        "class_weighting": None,
    },

    "probability_threshold": 0.13,

    "threshold_selection": {
        "evidence_predictor_years": "2015–2020",
        "objective": (
            "Highest precision among thresholds within "
            "99% of maximum pooled F2"
        ),
        "test_data_used": False,
    },

    "test_period": {
        "predictor_years": "2021–2022",
        "outcome_years": "2022–2023",
        "observations": 498,
        "shortages": 51,
    },

    "test_results": {
        "warnings_issued": 159,
        "shortages_detected": 33,
        "shortages_missed": 18,
        "false_warnings": 126,
        "precision": float(
            point_estimates["Precision"]
        ),
        "recall": float(
            point_estimates["Recall"]
        ),
        "specificity": float(
            point_estimates["Specificity"]
        ),
        "f2": float(
            point_estimates["F2"]
        ),
        "balanced_accuracy": float(
            point_estimates["Balanced accuracy"]
        ),
        "roc_auc": float(
            point_estimates["ROC-AUC"]
        ),
        "pr_auc": float(
            point_estimates["PR-AUC"]
        ),
        "brier_score": float(
            point_estimates["Brier score"]
        ),
        "pr_auc_no_skill_reference":
            float(test_event_rate),
        "pr_auc_lift":
            float(test_pr_auc_lift),
    },

    "interpretation": {
        "intended_use": (
            "Early-warning screening and prioritisation"
        ),
        "not_intended_for": (
            "Automatic declaration that a shortage "
            "will occur"
        ),
        "important_limitation": (
            "A substantial number of false warnings "
            "requires human review"
        ),
        "post_test_model_change": False,
        "post_test_threshold_change": False,
    },

    "software": {
        "scikit_learn_version":
            sklearn.__version__,
        "pandas_version":
            pd.__version__,
        "numpy_version":
            np.__version__,
    },
}

metadata_path = (
    model_output_directory
    / "shortage_final_evaluation_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(
        final_evaluation_metadata,
        metadata_file,
        indent=2,
        ensure_ascii=False,
    )

saved_file_records.append(
    {
        "Output":
            "shortage_final_evaluation_metadata.json",
        "Rows": np.nan,
        "Columns": np.nan,
        "File size MB":
            metadata_path.stat().st_size / (1024 ** 2),
    }
)

# ------------------------------------------------------------
# 5. Reload and validate the saved evidence
# ------------------------------------------------------------

reloaded_test_predictions = pd.read_parquet(
    test_predictions_path
)

reloaded_test_metrics = pd.read_csv(
    model_output_directory
    / "shortage_final_test_metrics.csv"
)

reloaded_decision_record = pd.read_csv(
    model_output_directory
    / "shortage_final_evaluation_record.csv"
)

with open(
    metadata_path,
    "r",
    encoding="utf-8",
) as metadata_file:
    reloaded_metadata = json.load(metadata_file)

assert len(reloaded_test_predictions) == 498

assert (
    reloaded_test_predictions[
        "shortage_next_year"
    ].sum()
    == 51
)

assert (
    reloaded_test_predictions[
        "predicted_shortage_warning"
    ].sum()
    == 159
)

assert np.allclose(
    reloaded_test_predictions[
        "predicted_shortage_probability"
    ].to_numpy(),
    test_predictions[
        "predicted_shortage_probability"
    ].to_numpy(),
)

assert reloaded_test_metrics.loc[
    0, "Threshold"
] == 0.13

assert (
    reloaded_metadata["probability_threshold"]
    == 0.13
)

assert (
    reloaded_metadata["interpretation"]
    ["post_test_model_change"]
    is False
)

assert (
    reloaded_metadata["interpretation"]
    ["post_test_threshold_change"]
    is False
)

# ------------------------------------------------------------
# 6. Display saved-output summary
# ------------------------------------------------------------

saved_evaluation_summary = pd.DataFrame(
    saved_file_records
)

print("Saved final evaluation evidence:")
display(
    saved_evaluation_summary.style.format(
        {"File size MB": "{:.4f}"}
    )
)

print("\nFinal evaluation decision record:")
display(final_evaluation_record)

print(
    "\nAll final locked-test results were saved, "
    "reloaded and validated successfully.\n"
    "No model or threshold decision was changed."
)

Saved final evaluation evidence:


,Output,Rows,Columns,File size MB
0,shortage_final_test_metrics.csv,1.000000,20.000000,0.0005
1,shortage_final_test_uncertainty.csv,8.000000,4.000000,0.0006
2,shortage_final_test_risk_bands.csv,4.000000,7.000000,0.0006
3,shortage_final_test_by_year.csv,2.000000,12.000000,0.0004
4,shortage_final_test_by_commodity.csv,8.000000,12.000000,0.0011
5,shortage_development_test_comparison.csv,2.000000,9.000000,0.0003
6,shortage_final_evaluation_record.csv,21.000000,2.000000,0.0008
7,shortage_final_test_missed_events.csv,18.000000,10.000000,0.0020
8,shortage_final_test_predictions.parquet,498.000000,10.000000,0.0124
9,shortage_final_evaluation_metadata.json,nan,nan,0.0016



Final evaluation decision record:


,Decision component,Decision
0,Model family,Random forest
1,Trees,500
2,Maximum depth,Unlimited
3,Minimum observations per leaf,3
4,Class weighting,None
5,Final probability threshold,0.1300
6,Threshold evidence,Out-of-time development predictions from 2015–...
7,Threshold selection rule,Highest precision among thresholds within 99% ...
8,Development predictor years,2010–2020
9,Final test predictor years,2021–2022



All final locked-test results were saved, reloaded and validated successfully.
No model or threshold decision was changed.


In [19]:
# ============================================================
# BUILD AND SAVE THE FINAL OPERATIONAL MODEL
#
# Evaluation model:
#   Trained through 2020 and judged on 2021–2022.
#
# Operational model:
#   Refit using every eligible predictor row from 2010–2022,
#   with known outcomes through 2023.
#
# The locked model design and 0.13 threshold remain unchanged.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import cloudpickle
import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone
from sklearn.pipeline import Pipeline

# ------------------------------------------------------------
# 1. Define file locations
# ------------------------------------------------------------

project_directory = Path(
    "/Users/adewale/Documents/food_security_predictor"
)

processed_data_directory = (
    project_directory
    / "data"
    / "processed"
    / "africa_first"
)

model_output_directory = (
    project_directory
    / "models"
    / "africa_first"
)

model_output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

model_ready_path = (
    processed_data_directory
    / "africa_model_ready_shortage_features_2010_2022.parquet"
)

assert model_ready_path.exists()

# ------------------------------------------------------------
# 2. Reload all eligible labelled observations
# ------------------------------------------------------------

operational_training_data = pd.read_parquet(
    model_ready_path
)

operational_feature_names = list(
    X_test_locked.columns
)

target_column = "shortage_next_year"

required_columns = (
    operational_feature_names
    + [
        target_column,
        "Year",
        "target_year",
        "Temporal split",
        "Area",
        "Item Code",
        "Item",
    ]
)

missing_required_columns = sorted(
    set(required_columns)
    - set(operational_training_data.columns)
)

assert not missing_required_columns, (
    "Required operational-training columns are missing: "
    f"{missing_required_columns}"
)

X_operational = operational_training_data[
    operational_feature_names
].copy()

y_operational = (
    operational_training_data[target_column]
    .astype(int)
    .copy()
)

# ------------------------------------------------------------
# 3. Validate the complete training history
# ------------------------------------------------------------

operational_training_summary = pd.DataFrame(
    {
        "Result": [
            "Operational training rows",
            "Raw predictors",
            "Countries",
            "Commodities",
            "Country-commodity pairs",
            "Predictor year start",
            "Predictor year end",
            "Outcome year start",
            "Outcome year end",
            "Positive shortage events",
            "Negative outcomes",
            "Event rate %",
            "Missing target values",
            "Duplicate country-commodity-year keys",
            "Locked probability threshold",
        ],
        "Value": [
            len(operational_training_data),
            len(operational_feature_names),
            operational_training_data["Area"].nunique(),
            operational_training_data["Item Code"].nunique(),
            operational_training_data[
                ["Area", "Item Code"]
            ].drop_duplicates().shape[0],
            operational_training_data["Year"].min(),
            operational_training_data["Year"].max(),
            operational_training_data["target_year"].min(),
            operational_training_data["target_year"].max(),
            int(y_operational.sum()),
            int((y_operational == 0).sum()),
            100 * y_operational.mean(),
            int(y_operational.isna().sum()),
            int(
                operational_training_data.duplicated(
                    ["Area", "Item Code", "Year"]
                ).sum()
            ),
            final_probability_threshold,
        ],
    }
)

print("Operational model training summary:")
display(
    operational_training_summary.style.format(
        {"Value": "{:,.4f}"}
    )
)

assert len(operational_training_data) == 3248
assert len(operational_feature_names) == 129
assert operational_training_data["Year"].min() == 2010
assert operational_training_data["Year"].max() == 2022
assert operational_training_data["target_year"].min() == 2011
assert operational_training_data["target_year"].max() == 2023
assert y_operational.sum() == 334
assert y_operational.isna().sum() == 0
assert set(y_operational.unique()) == {0, 1}
assert final_probability_threshold == 0.13

assert not operational_training_data.duplicated(
    ["Area", "Item Code", "Year"]
).any()

# Confirm that all three historical periods are now included.
operational_split_summary = (
    operational_training_data
    .groupby(
        "Temporal split",
        observed=True,
    )
    .agg(
        Rows=(target_column, "size"),
        Positive_events=(target_column, "sum"),
        Predictor_year_start=("Year", "min"),
        Predictor_year_end=("Year", "max"),
        Target_year_start=("target_year", "min"),
        Target_year_end=("target_year", "max"),
    )
    .reset_index()
)

operational_split_summary[
    "Negative_events"
] = (
    operational_split_summary["Rows"]
    - operational_split_summary["Positive_events"]
)

operational_split_summary[
    "Event_rate_pct"
] = (
    100
    * operational_split_summary["Positive_events"]
    / operational_split_summary["Rows"]
)

print("\nHistorical periods included in the operational fit:")
display(
    operational_split_summary.style.format(
        {"Event_rate_pct": "{:.2f}"}
    )
)

# ------------------------------------------------------------
# 4. Rebuild the same locked modelling system
# ------------------------------------------------------------
# clone() creates a fresh copy of the already-selected model
# design. It does not reuse fitted information from the test.
#
# The fresh copy is then fitted using all 3,248 observations.

if hasattr(final_model, "named_steps"):

    # The earlier final model already contains preprocessing.
    operational_pipeline = clone(final_model)

else:

    assert "final_preprocessor" in globals(), (
        "The fitted final_preprocessor is unavailable. "
        "Do not invent a replacement; restore the previous "
        "Notebook 4 model-lock cell."
    )

    operational_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                clone(final_preprocessor),
            ),
            (
                "random_forest",
                clone(final_model),
            ),
        ]
    )

# Fit once using all eligible historical observations.
operational_pipeline.fit(
    X_operational,
    y_operational,
)

# ------------------------------------------------------------
# 5. Basic technical validation
# ------------------------------------------------------------

sample_probabilities = (
    operational_pipeline.predict_proba(
        X_operational.iloc[:50]
    )[:, 1]
)

assert len(sample_probabilities) == 50
assert np.isfinite(sample_probabilities).all()
assert (
    (sample_probabilities >= 0)
    & (sample_probabilities <= 1)
).all()

# This is only a technical check that predictions can be made.
# It is not a new performance evaluation.
sample_warnings = (
    sample_probabilities
    >= final_probability_threshold
).astype(int)

technical_check = pd.DataFrame(
    {
        "Result": [
            "Sample rows scored successfully",
            "Finite probabilities",
            "Probabilities within zero and one",
            "Threshold retained",
            "Test results reused as training-performance claims",
        ],
        "Value": [
            len(sample_probabilities),
            bool(np.isfinite(sample_probabilities).all()),
            bool(
                (
                    (sample_probabilities >= 0)
                    & (sample_probabilities <= 1)
                ).all()
            ),
            final_probability_threshold,
            False,
        ],
    }
)

print("\nOperational-model technical check:")
display(technical_check)

# ------------------------------------------------------------
# 6. Create a feature-schema record
# ------------------------------------------------------------

operational_feature_schema = pd.DataFrame(
    {
        "Feature order":
            np.arange(
                1,
                len(operational_feature_names) + 1,
            ),
        "Feature":
            operational_feature_names,
        "Saved data type": [
            str(X_operational[column].dtype)
            for column in operational_feature_names
        ],
    }
)

feature_name_signature = hashlib.sha256(
    "\n".join(
        operational_feature_names
    ).encode("utf-8")
).hexdigest()

# ------------------------------------------------------------
# 7. Create the portable model bundle
# ------------------------------------------------------------
# The bundle contains:
# - preprocessing rules,
# - fitted random forest,
# - feature order,
# - probability threshold,
# - clear usage information.

operational_model_bundle = {
    "model_name":
        "Africa food-supply shortage early-warning model",

    "model_version":
        "1.0",

    "fitted_pipeline":
        operational_pipeline,

    "feature_names":
        operational_feature_names,

    "feature_name_signature_sha256":
        feature_name_signature,

    "target_column":
        target_column,

    "probability_threshold":
        float(final_probability_threshold),

    "risk_interpretation": {
        "below_0_05":
            "Very low modelled risk",
        "0_05_to_below_0_13":
            "Watch only; not a formal warning",
        "0_13_to_below_0_30":
            "Shortage warning requiring review",
        "0_30_and_above":
            "High-priority shortage warning",
    },

    "training_scope": {
        "rows": int(len(operational_training_data)),
        "predictor_year_start": 2010,
        "predictor_year_end": 2022,
        "outcome_year_start": 2011,
        "outcome_year_end": 2023,
        "positive_events": int(y_operational.sum()),
        "countries": int(
            operational_training_data["Area"].nunique()
        ),
        "commodities": int(
            operational_training_data[
                "Item Code"
            ].nunique()
        ),
    },

    "locked_design": {
        "model_family": "Random forest",
        "trees": 500,
        "maximum_depth": None,
        "minimum_observations_per_leaf": 3,
        "class_weighting": None,
        "probability_threshold": 0.13,
    },

    "evaluation_status": {
        "final_test_completed": True,
        "reported_test_predictor_years": "2021–2022",
        "reported_test_outcome_years": "2022–2023",
        "test_results_saved_separately": True,
        "operational_refit_used_test_rows": True,
        "operational_model_must_not_be_retested_on_2021_2022":
            True,
    },

    "intended_use": (
        "Early-warning screening and prioritisation "
        "of country-commodity observations for human review."
    ),

    "not_intended_for": (
        "Automatic declaration that a food shortage "
        "will occur."
    ),

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "software_versions": {
        "scikit_learn": sklearn.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "cloudpickle": cloudpickle.__version__,
    },
}

# ------------------------------------------------------------
# 8. Save the operational model
# ------------------------------------------------------------

operational_model_path = (
    model_output_directory
    / "africa_shortage_operational_model_2010_2022.pkl"
)

with open(
    operational_model_path,
    "wb",
) as model_file:
    cloudpickle.dump(
        operational_model_bundle,
        model_file,
    )

# Calculate a fingerprint for the exact saved file.
# If the file changes later, its fingerprint will also change.

def calculate_sha256(file_path):
    sha256_hash = hashlib.sha256()

    with open(file_path, "rb") as input_file:
        for block in iter(
            lambda: input_file.read(1024 * 1024),
            b"",
        ):
            sha256_hash.update(block)

    return sha256_hash.hexdigest()

model_file_signature = calculate_sha256(
    operational_model_path
)

# ------------------------------------------------------------
# 9. Save the supporting records
# ------------------------------------------------------------

feature_schema_path = (
    model_output_directory
    / "shortage_operational_feature_schema.csv"
)

training_summary_path = (
    model_output_directory
    / "shortage_operational_training_summary.csv"
)

split_summary_path = (
    model_output_directory
    / "shortage_operational_split_summary.csv"
)

manifest_path = (
    model_output_directory
    / "shortage_operational_model_manifest.json"
)

operational_feature_schema.to_csv(
    feature_schema_path,
    index=False,
)

operational_training_summary.to_csv(
    training_summary_path,
    index=False,
)

operational_split_summary.to_csv(
    split_summary_path,
    index=False,
)

operational_manifest = {
    key: value
    for key, value in operational_model_bundle.items()
    if key != "fitted_pipeline"
}

operational_manifest["model_file"] = (
    operational_model_path.name
)

operational_manifest[
    "model_file_signature_sha256"
] = model_file_signature

operational_manifest[
    "model_file_size_mb"
] = (
    operational_model_path.stat().st_size
    / (1024 ** 2)
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        operational_manifest,
        manifest_file,
        indent=2,
        ensure_ascii=False,
    )

# ------------------------------------------------------------
# 10. Reload and validate the saved model
# ------------------------------------------------------------

with open(
    operational_model_path,
    "rb",
) as model_file:
    reloaded_operational_bundle = (
        cloudpickle.load(model_file)
    )

reloaded_pipeline = (
    reloaded_operational_bundle[
        "fitted_pipeline"
    ]
)

reloaded_probabilities = (
    reloaded_pipeline.predict_proba(
        X_operational.iloc[:50]
    )[:, 1]
)

assert np.allclose(
    sample_probabilities,
    reloaded_probabilities,
)

assert (
    reloaded_operational_bundle[
        "probability_threshold"
    ]
    == 0.13
)

assert (
    reloaded_operational_bundle[
        "feature_names"
    ]
    == operational_feature_names
)

assert (
    reloaded_operational_bundle[
        "feature_name_signature_sha256"
    ]
    == feature_name_signature
)

assert (
    calculate_sha256(
        operational_model_path
    )
    == model_file_signature
)

# ------------------------------------------------------------
# 11. Display the saved operational outputs
# ------------------------------------------------------------

saved_operational_outputs = pd.DataFrame(
    [
        {
            "Output":
                "Operational model bundle",
            "File":
                operational_model_path.name,
            "File size MB":
                operational_model_path.stat().st_size
                / (1024 ** 2),
        },
        {
            "Output":
                "Feature schema",
            "File":
                feature_schema_path.name,
            "File size MB":
                feature_schema_path.stat().st_size
                / (1024 ** 2),
        },
        {
            "Output":
                "Training summary",
            "File":
                training_summary_path.name,
            "File size MB":
                training_summary_path.stat().st_size
                / (1024 ** 2),
        },
        {
            "Output":
                "Historical split summary",
            "File":
                split_summary_path.name,
            "File size MB":
                split_summary_path.stat().st_size
                / (1024 ** 2),
        },
        {
            "Output":
                "Operational model manifest",
            "File":
                manifest_path.name,
            "File size MB":
                manifest_path.stat().st_size
                / (1024 ** 2),
        },
    ]
)

print("\nSaved operational-model outputs:")
display(
    saved_operational_outputs.style.format(
        {"File size MB": "{:.4f}"}
    )
)

print(
    "\nThe final operational model was fitted using all "
    "3,248 eligible historical observations.\n"
    "It was saved, reloaded and validated successfully.\n"
    "The locked model design and 0.13 probability threshold "
    "were retained unchanged.\n"
    "The previously reported final-test results remain the "
    "official unbiased evaluation."
)

Operational model training summary:


,Result,Value
0,Operational training rows,"3,248.0000"
1,Raw predictors,129.0000
2,Countries,43.0000
3,Commodities,8.0000
4,Country-commodity pairs,260.0000
5,Predictor year start,"2,010.0000"
6,Predictor year end,"2,022.0000"
7,Outcome year start,"2,011.0000"
8,Outcome year end,"2,023.0000"
9,Positive shortage events,334.0000



Historical periods included in the operational fit:


,Temporal split,Rows,Positive_events,Predictor_year_start,Predictor_year_end,Target_year_start,Target_year_end,Negative_events,Event_rate_pct
0,Test,498,51,2021,2022,2022,2023,447,10.24
1,Train,2250,231,2010,2018,2011,2019,2019,10.27
2,Validation,500,52,2019,2020,2020,2021,448,10.40



Operational-model technical check:


,Result,Value
0,Sample rows scored successfully,50
1,Finite probabilities,True
2,Probabilities within zero and one,True
3,Threshold retained,0.1300
4,Test results reused as training-performance cl...,False



Saved operational-model outputs:


,Output,File,File size MB
0,Operational model bundle,africa_shortage_operational_model_2010_2022.pkl,13.8077
1,Feature schema,shortage_operational_feature_schema.csv,0.0047
2,Training summary,shortage_operational_training_summary.csv,0.0004
3,Historical split summary,shortage_operational_split_summary.csv,0.0003
4,Operational model manifest,shortage_operational_model_manifest.json,0.0061



The final operational model was fitted using all 3,248 eligible historical observations.
It was saved, reloaded and validated successfully.
The locked model design and 0.13 probability threshold were retained unchanged.
The previously reported final-test results remain the official unbiased evaluation.
